In [ ]:
# Cell 1 — Imports + settings  # 3.12.26 revised for new dataset + centralized imports

import os  # 3.12.26 file and environment utilities used across later save/load cells
import re  # 3.12.26 regular expressions used later for parsing artifact paths
import sys  # 3.12.26 Python version capture used later when saving run metadata
import json  # 3.12.26 save configs and metrics to json files
import time  # 3.12.26 epoch timing in the training utilities cell
import random  # 3.12.26 seed Python randomness for reproducible splits and training
import pickle  # 3.12.26 save full Python objects such as histories
import tarfile  # 3.12.26 archive saved artifacts at the end of the notebook
import shutil  # 3.12.26 copy saved figures and folders into snapshot outputs
from bisect import bisect_right  # 3.12.26 map global stimulus indices back to each run
from datetime import datetime  # 3.12.26 timestamp saved outputs and summary files
from pathlib import Path  # 3.12.26 use clear and safe filesystem paths
from IPython.display import display  # 3.12.26 display summary tables in later evaluation cells

import numpy as np  # 3.12.26 arrays, indexing, and numeric helpers throughout the notebook
import pandas as pd  # 3.12.26 load and manipulate metadata tables
import zarr  # 3.12.26 open the saved retina response dataset
import torch  # 3.12.26 core PyTorch package for tensors and training
import torch.nn as nn  # 3.12.26 neural network layers and loss wrappers
import sklearn  # 3.12.26 save sklearn version information with notebook outputs
import matplotlib  # 3.12.26 save matplotlib version information with notebook outputs
import matplotlib.pyplot as plt  # 3.12.26 make training and evaluation figures
from PIL import Image  # 3.12.26 save misclassified stimulus images
from torch.utils.data import Dataset, DataLoader  # 3.12.26 dataset class and minibatch loading
from sklearn.model_selection import StratifiedShuffleSplit, ShuffleSplit  # 3.12.26 balanced splitting helpers used later
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report  # 3.12.26 evaluation metrics and displays

SEED = 5  # 0 3.12.26 single master seed for the notebook
SPLIT_SEED = SEED  # 3.12.26 seed used when creating train, val, and test splits
INIT_SEED_BASE = SEED  # 3.12.26 base seed used for model weight initialization trials
SHUFFLE_SEED_BASE = SEED  # 3.12.26 base seed used for DataLoader shuffle order

SAVE_VAL_ARTIFACT_EPOCHS = {2, 5}  # 3.12.26 save validation artifacts on these epochs
EXPORT_DIR = Path("/data/retina-neitz/notebooks/export_retina_20260313_230715")  # 3.12.26 point to the new retina export folder
ZARR_PATH = EXPORT_DIR / "dataset.zarr"  # 3.12.26 point to the new dataset zarr store
META_CSV = EXPORT_DIR / "metadata.csv"  # 3.12.26 point to the new metadata table
ARTIFACTS_DIR = EXPORT_DIR / "cnn_artifacts"  # 3.12.26 keep CNN artifacts inside the new export folder
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)  # 3.12.26 make sure the artifact folder exists before saving anything

RUNS = [  # 3.12.26 keep one clear run record for the new dataset only
    {  # 3.12.26 single-run setup for this notebook version
        "name": "export_retina_20260313_230715",  # 3.12.26 descriptive run name for prints and saved outputs
        "zarr_path": ZARR_PATH,  # 3.12.26 zarr path for the new dataset
        "meta_csv": META_CSV,  # 3.12.26 metadata path for the new dataset
    },  # 3.12.26 end single-run record
]  # 3.12.26 end run list

assert ZARR_PATH.exists(), f"Zarr path not found: {ZARR_PATH}"  # 3.12.26 fail early if the dataset file is missing
assert META_CSV.exists(), f"Metadata CSV not found: {META_CSV}"  # 3.12.26 fail early if the metadata file is missing
print("Found Zarr:", ZARR_PATH)  # 3.12.26 confirm the new zarr path
print("Found CSV :", META_CSV)  # 3.12.26 confirm the new metadata path

CLASS_ORDER = ["gray_d", "gray_l", "red", "green", "blue", "yellow"]  # 3.12.26 use only the new dataset labels
label_to_int = {label: i for i, label in enumerate(CLASS_ORDER)}  # 3.12.26 fixed label-to-index mapping for the CNN
int_to_label = {i: label for label, i in label_to_int.items()}  # 3.12.26 inverse mapping for reports and plots

TARGET_CLASS_WEIGHTS = {  # 3.12.26 keep the same intended class mix but with new gray labels
    "gray_d": 25.0,  # 3.12.26 weight for dark-gray control stimuli
    "gray_l": 25.0,  # 3.12.26 weight for light-gray control stimuli
    "red": 12.5,  # 3.12.26 weight for red stimuli
    "green": 12.5,  # 3.12.26 weight for green stimuli
    "blue": 12.5,  # 3.12.26 weight for blue stimuli
    "yellow": 12.5,  # 3.12.26 weight for yellow stimuli
}  # 3.12.26 end target class weights
_target_total = float(sum(TARGET_CLASS_WEIGHTS.values()))  # 3.12.26 total weight used to normalize class fractions
TARGET_CLASS_FRAC = {label: weight / _target_total for label, weight in TARGET_CLASS_WEIGHTS.items()}  # 3.12.26 normalized target fractions used during subsampling

CHANNEL_GROUPS = {  # 3.12.26 define the available channel sets in one place
    "Classic4": ["l_on", "l_off", "m_on", "m_off"],  # 3.12.26 classic four-channel group
    "Classic4_plus_s": ["l_on", "l_off", "m_on", "m_off", "s_on"],  # 3.12.26 classic group plus S-ON
    "Neitz4": ["l_h2_on", "m_h2_on", "l_h2_off", "m_h2_off"],  # 3.12.26 H2-associated four-channel group
    "Neitz8": ["l_on", "l_off", "m_on", "m_off", "l_h2_on", "m_h2_on", "l_h2_off", "m_h2_off"],  # 3.12.26 combined classic and H2 channels
    "All9": ["l_on", "l_off", "m_on", "m_off", "l_h2_on", "m_h2_on", "l_h2_off", "m_h2_off", "s_on"],  # 3.12.26 all available channels together
}  # 3.12.26 end channel group definitions
BUILD_GROUPS = ["Classic4", "Classic4_plus_s", "Neitz8"]  # 3.12.26 keep the main comparison groups for this notebook

def set_all_seeds(seed: int) -> None:  # 3.12.26 simple reproducibility helper used across the notebook
    seed = int(seed)  # 3.12.26 make sure every library receives a plain integer seed
    os.environ["PYTHONHASHSEED"] = str(seed)  # 3.12.26 reduce Python hash randomness between runs
    random.seed(seed)  # 3.12.26 seed Python random
    np.random.seed(seed)  # 3.12.26 seed NumPy random
    torch.manual_seed(seed)  # 3.12.26 seed PyTorch on CPU
    if torch.cuda.is_available():  # 3.12.26 only seed CUDA when a GPU exists
        torch.cuda.manual_seed_all(seed)  # 3.12.26 seed all available CUDA devices
    torch.backends.cudnn.deterministic = True  # 3.12.26 prefer deterministic cuDNN behavior
    torch.backends.cudnn.benchmark = False  # 3.12.26 turn off cuDNN autotuning for repeatability
    try:  # 3.12.26 use warn_only when this PyTorch version supports it
        torch.use_deterministic_algorithms(True, warn_only=True)  # 3.12.26 ask PyTorch for deterministic algorithms without hard failure
    except TypeError:  # 3.12.26 older PyTorch versions may not support warn_only
        torch.use_deterministic_algorithms(True)  # 3.12.26 still request deterministic algorithms

def get_torch_rng_state() -> dict:  # 3.12.26 save exact torch RNG state before init-sensitive steps
    state = {"cpu": torch.get_rng_state()}  # 3.12.26 capture CPU RNG state
    if torch.cuda.is_available():  # 3.12.26 only capture CUDA state when a GPU exists
        state["cuda"] = torch.cuda.get_rng_state_all()  # 3.12.26 capture RNG state for every CUDA device
    return state  # 3.12.26 return the RNG snapshot

def set_torch_rng_state(state: dict) -> None:  # 3.12.26 restore an RNG snapshot saved earlier
    torch.set_rng_state(state["cpu"])  # 3.12.26 restore CPU RNG state
    if torch.cuda.is_available() and ("cuda" in state):  # 3.12.26 restore CUDA RNG state only when available
        torch.cuda.set_rng_state_all(state["cuda"])  # 3.12.26 restore RNG state for every CUDA device

def make_torch_generator(seed: int) -> torch.Generator:  # 3.12.26 create a dedicated generator for deterministic shuffling
    generator = torch.Generator()  # 3.12.26 make a standalone torch generator
    generator.manual_seed(int(seed))  # 3.12.26 seed that generator
    return generator  # 3.12.26 return the seeded generator

def seed_worker(worker_id: int) -> None:  # 3.12.26 seed NumPy and Python inside each DataLoader worker
    worker_seed = torch.initial_seed() % (2**32)  # 3.12.26 derive a stable worker seed from torch
    np.random.seed(worker_seed)  # 3.12.26 seed NumPy inside the worker
    random.seed(worker_seed)  # 3.12.26 seed Python random inside the worker

set_all_seeds(SEED)  # 3.12.26 apply reproducible seeds once at notebook start
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # 3.12.26 use GPU when available and CPU otherwise
print("Using device:", device)  # 3.12.26 confirm the chosen device

In [ ]:
# Cell 2 — Load data from Zarr + metadata checks  # 3.12.26 simplified for one exact dataset and one exact Zarr layout

meta = pd.read_csv(META_CSV).reset_index(drop=True)  # 3.12.26 load metadata.csv and reset to a clean row index
root = zarr.open_group(str(ZARR_PATH), mode="r")  # 3.12.26 open the dataset.zarr store in read-only mode

assert "imgs" in root, "dataset.zarr must contain top-level array 'imgs'"  # 3.12.26 require the exact stimulus-image array name
assert "derived" in root, "dataset.zarr must contain group 'derived'"  # 3.12.26 require the derived products group
assert "outs_fill" in root["derived"], "dataset.zarr must contain derived/outs_fill"  # 3.12.26 require the filled response maps

imgs = root["imgs"]  # 3.12.26 use the exact stimulus-image array created in the export notebook
outs_fill = root["derived"]["outs_fill"]  # 3.12.26 use the exact filled-response array path confirmed from your printout
keys_order = [str(k) for k in root.attrs["RESPONSE_KEYS"]]  # 3.12.26 use the exact channel-name attribute saved in the Zarr store

N, C, H, W = outs_fill.shape  # 3.12.26 read the filled-response array shape
IMG_SHAPE = tuple(int(x) for x in imgs.shape[1:])  # 3.12.26 store the per-stimulus image shape

assert imgs.shape[0] == N, f"Mismatch between imgs and outs_fill: imgs={imgs.shape[0]} outs_fill={N}"  # 3.12.26 require equal numbers of images and response maps
assert len(meta) == N, f"Mismatch between metadata rows and outs_fill: meta={len(meta)} outs_fill={N}"  # 3.12.26 require metadata rows to match the dataset
assert len(keys_order) == C, f"Mismatch: outs_fill has C={C} channels but RESPONSE_KEYS has {len(keys_order)} entries"  # 3.12.26 require the saved channel names to match the data

required_meta_cols = ["stim_ID", "true_label", "obj_int", "obj_sat", "bg_int", "bg_sat", "bg_hue"]  # 3.12.26 require the exact metadata columns written by the stimulus-generation notebook
missing_meta_cols = [col for col in required_meta_cols if col not in meta.columns]  # 3.12.26 collect any missing required columns
assert len(missing_meta_cols) == 0, f"metadata.csv is missing required columns: {missing_meta_cols}"  # 3.12.26 stop immediately if the metadata schema is wrong

meta["stim_ID"] = pd.to_numeric(meta["stim_ID"], errors="coerce")  # 3.12.26 force stim_ID to numeric so row alignment can be checked
assert meta["stim_ID"].notna().all(), "Found NaNs or non-numeric values in meta['stim_ID']"  # 3.12.26 require every stimulus ID to be valid
meta["stim_ID"] = meta["stim_ID"].astype(int)  # 3.12.26 store stim_ID as integers after validation
assert np.array_equal(meta["stim_ID"].to_numpy(), np.arange(N)), "meta['stim_ID'] must run from 0 to N-1 in dataset order"  # 3.12.26 require metadata row order to match the Zarr arrays exactly

meta["stimulus_index"] = meta["stim_ID"].astype(int)  # 3.12.26 create one simple canonical global index column
meta["stim_id"] = meta["stimulus_index"].astype(int)  # 3.12.26 keep the shorter stim_id name used later in the notebook
meta["stim_id_local"] = meta["stimulus_index"].astype(int)  # 3.12.26 local index is the same as global index in this single-run notebook

meta["hue"] = meta["true_label"].astype(str)  # 3.12.26 standardize class labels to one simple internal column name
meta["intensity"] = pd.to_numeric(meta["obj_int"], errors="coerce").astype(float)  # 3.12.26 standardize object intensity to one internal column
meta["saturation"] = pd.to_numeric(meta["obj_sat"], errors="coerce").astype(float)  # 3.12.26 standardize object saturation to one internal column
meta["bg_intensity"] = pd.to_numeric(meta["bg_int"], errors="coerce").astype(float)  # 3.12.26 standardize background intensity to one internal column
meta["bg_saturation"] = pd.to_numeric(meta["bg_sat"], errors="coerce").astype(float)  # 3.12.26 standardize background saturation to one internal column
meta["bg_hue"] = meta["bg_hue"].astype(str)  # 3.12.26 store background hue as strings

assert meta["hue"].notna().all(), "Found NaNs in meta['hue']"  # 3.12.26 require valid class labels
assert meta["intensity"].notna().all(), "Found NaNs or non-numeric values in meta['intensity']"  # 3.12.26 require valid object intensities
assert meta["saturation"].notna().all(), "Found NaNs or non-numeric values in meta['saturation']"  # 3.12.26 require valid object saturations
assert meta["bg_intensity"].notna().all(), "Found NaNs or non-numeric values in meta['bg_intensity']"  # 3.12.26 require valid background intensities
assert meta["bg_saturation"].notna().all(), "Found NaNs or non-numeric values in meta['bg_saturation']"  # 3.12.26 require valid background saturations
assert meta["bg_hue"].notna().all(), "Found NaNs in meta['bg_hue']"  # 3.12.26 require valid background hue labels

expected_hues = set(CLASS_ORDER)  # 3.12.26 the only allowed class labels are the six labels defined in Cell 1
hues_found = set(meta["hue"].unique())  # 3.12.26 collect the labels actually present in the metadata
unknown_hues = sorted(hues_found - expected_hues)  # 3.12.26 detect labels that should not exist in this notebook
missing_hues = sorted(expected_hues - hues_found)  # 3.12.26 detect expected labels that are absent
assert len(unknown_hues) == 0, f"Found unexpected class labels: {unknown_hues}"  # 3.12.26 stop if labels do not match the new dataset labels

EPS_GRAY = 1e-8  # 3.12.26 small tolerance for treating saturation as zero
meta["is_gray_control"] = meta["hue"].isin(["gray_d", "gray_l"])  # 3.12.26 mark the two gray control classes explicitly
meta["is_grayscale"] = meta["is_gray_control"] | (meta["saturation"] <= EPS_GRAY)  # 3.12.26 mark grayscale stimuli
meta["is_color"] = ~meta["is_grayscale"]  # 3.12.26 mark color stimuli
meta["bg_is_grayscale"] = (meta["bg_saturation"] <= EPS_GRAY) | meta["bg_hue"].isin(["gray", "grey"])  # 3.12.26 mark grayscale backgrounds
meta["bg_is_color"] = ~meta["bg_is_grayscale"]  # 3.12.26 mark color backgrounds

global_meta = {  # 3.12.26 save one compact dataset summary for later output files
    "export_dir": str(EXPORT_DIR),  # 3.12.26 record the export directory used for this run
    "runs": [  # 3.12.26 keep a single run record because this notebook now uses one dataset only
        {  # 3.12.26 summary record for this dataset
            "name": str(RUNS[0]["name"]),  # 3.12.26 save the run name
            "zarr_path": str(ZARR_PATH),  # 3.12.26 save the exact Zarr path
            "meta_csv": str(META_CSV),  # 3.12.26 save the exact metadata path
            "img_key": "imgs",  # 3.12.26 record the exact image-array name
            "outs_fill_key": "derived/outs_fill",  # 3.12.26 record the exact filled-response array path
            "start": 0,  # 3.12.26 single-run dataset starts at index 0
            "N": int(N),  # 3.12.26 number of stimuli in this dataset
        }
    ],
    "N": int(N),  # 3.12.26 total number of stimuli
    "C": int(C),  # 3.12.26 number of response channels
    "H": int(H),  # 3.12.26 response-map height
    "W": int(W),  # 3.12.26 response-map width
    "img_shape": tuple(IMG_SHAPE),  # 3.12.26 per-stimulus image shape
    "keys_order": list(keys_order),  # 3.12.26 response channel names
    "class_order": list(CLASS_ORDER),  # 3.12.26 class-label order used by this notebook
    "hue_counts": meta["hue"].value_counts().to_dict(),  # 3.12.26 class counts in this dataset
}  # 3.12.26 end dataset summary dictionary

def split_global_idx(i: int):  # 3.12.26 simple helper because this notebook uses one dataset only
    return 0, int(i)  # 3.12.26 every valid index belongs to run 0 with the same local index

def get_outs_fill_sample(i: int) -> np.ndarray:  # 3.12.26 lazy response accessor used later by the Dataset class
    return np.asarray(outs_fill[int(i), :, :, :], dtype=np.float32)  # 3.12.26 load one filled response sample as float32

def get_img_sample(i: int) -> np.ndarray:  # 3.12.26 lazy image accessor used later when saving stimulus images
    return np.asarray(imgs[int(i), :, :, :], dtype=np.float32)  # 3.12.26 load one image sample as float32

print("Total N:", N)  # 3.12.26 print total stimulus count
print("outs_fill per-sample shape:", (C, H, W))  # 3.12.26 print one filled-response sample shape
print("imgs per-sample shape:", IMG_SHAPE)  # 3.12.26 print one image sample shape
print("Channels (C):", C)  # 3.12.26 print the number of response channels
print("keys_order:", keys_order)  # 3.12.26 print the response channel names
print("Hue counts:\n", meta["hue"].value_counts())  # 3.12.26 print class counts
print("Grayscale count:", int(meta["is_grayscale"].sum()))  # 3.12.26 print the number of grayscale stimuli
print("Gray-control count:", int(meta["is_gray_control"].sum()))  # 3.12.26 print the number of gray control stimuli
print("Color-background count:", int(meta["bg_is_color"].sum()))  # 3.12.26 print the number of stimuli with color backgrounds
if missing_hues:  # 3.12.26 print this only if any expected classes are absent
    print("Note: missing expected hues in this dataset:", missing_hues)  # 3.12.26 helpful summary message

In [ ]:
# Cell 2b — Build condition masks (global indices)  # 3.12.26 simplified to match the new dataset labels

mask_is_grayscale = meta["is_grayscale"].to_numpy(dtype=bool)  # 3.12.26 grayscale stimuli
mask_is_color = meta["is_color"].to_numpy(dtype=bool)  # 3.12.26 color stimuli
mask_is_gray_control = meta["is_gray_control"].to_numpy(dtype=bool)  # 3.12.26 gray control stimuli only
mask_bg_is_grayscale = meta["bg_is_grayscale"].to_numpy(dtype=bool)  # 3.12.26 grayscale-background stimuli
mask_bg_is_color = meta["bg_is_color"].to_numpy(dtype=bool)  # 3.12.26 color-background stimuli

assert mask_is_grayscale.shape[0] == N, "mask_is_grayscale length must equal N"  # 3.12.26 sanity check
assert mask_is_color.shape[0] == N, "mask_is_color length must equal N"  # 3.12.26 sanity check
assert mask_is_gray_control.shape[0] == N, "mask_is_gray_control length must equal N"  # 3.12.26 sanity check
assert mask_bg_is_grayscale.shape[0] == N, "mask_bg_is_grayscale length must equal N"  # 3.12.26 sanity check
assert mask_bg_is_color.shape[0] == N, "mask_bg_is_color length must equal N"  # 3.12.26 sanity check

idx_all = np.arange(N, dtype=np.int64)  # 3.12.26 all stimulus indices
idx_grayscale = np.where(mask_is_grayscale)[0].astype(np.int64)  # 3.12.26 grayscale stimulus indices
idx_color = np.where(mask_is_color)[0].astype(np.int64)  # 3.12.26 color stimulus indices
idx_gray_control = np.where(mask_is_gray_control)[0].astype(np.int64)  # 3.12.26 gray control stimulus indices
idx_bg_grayscale = np.where(mask_bg_is_grayscale)[0].astype(np.int64)  # 3.12.26 grayscale-background stimulus indices
idx_bg_color = np.where(mask_bg_is_color)[0].astype(np.int64)  # 3.12.26 color-background stimulus indices

COND = {  # 3.12.26 single registry of masks and indices for later cells
    "mask": {  # 3.12.26 boolean masks of length N
        "all": np.ones(N, dtype=bool),  # 3.12.26 all stimuli
        "is_grayscale": mask_is_grayscale,  # 3.12.26 grayscale stimuli
        "is_color": mask_is_color,  # 3.12.26 color stimuli
        "is_gray_control": mask_is_gray_control,  # 3.12.26 gray control stimuli
        "bg_is_grayscale": mask_bg_is_grayscale,  # 3.12.26 grayscale backgrounds
        "bg_is_color": mask_bg_is_color,  # 3.12.26 color backgrounds
    },  # 3.12.26 end mask dictionary
    "idx": {  # 3.12.26 integer index arrays
        "all": idx_all,  # 3.12.26 all stimulus indices
        "grayscale": idx_grayscale,  # 3.12.26 grayscale stimulus indices
        "color": idx_color,  # 3.12.26 color stimulus indices
        "gray_control": idx_gray_control,  # 3.12.26 gray control stimulus indices
        "bg_grayscale": idx_bg_grayscale,  # 3.12.26 grayscale-background stimulus indices
        "bg_color": idx_bg_color,  # 3.12.26 color-background stimulus indices
    },  # 3.12.26 end index dictionary
}  # 3.12.26 end condition registry

print("COND sizes:", {k: int(v.size) for k, v in COND["idx"].items()})  # 3.12.26 quick count summary
print("COND checks:", {  # 3.12.26 quick sanity checks
    "grayscale+color==N": int(COND["idx"]["grayscale"].size + COND["idx"]["color"].size) == int(N),  # 3.12.26 partition check
    "gray_control<=grayscale": int(COND["idx"]["gray_control"].size) <= int(COND["idx"]["grayscale"].size),  # 3.12.26 subset check
    "bg_grayscale+bg_color==N": int(COND["idx"]["bg_grayscale"].size + COND["idx"]["bg_color"].size) == int(N),  # 3.12.26 background partition check
})  # 3.12.26 end sanity summary

In [ ]:
# Cell 3 — Encode labels + make weighted train/val/test splits with exact per-bin proportional allocation  # 03.14.2026 update header to reflect weighted pool + per-bin split fix

rng = np.random.default_rng(int(SPLIT_SEED))  # 3.12.26 reproducible random generator for all split steps

y_str = meta["hue"].astype(str).to_numpy()  # 3.12.26 class labels as strings in dataset order
unknown_hues = sorted(set(y_str) - set(CLASS_ORDER))  # 3.12.26 detect labels outside the six-class system
assert len(unknown_hues) == 0, f"Unknown hue labels found: {unknown_hues}"  # 3.12.26 fail early if labels are unexpected

y = np.array([label_to_int[h] for h in y_str], dtype=np.int64)  # 3.12.26 deterministic integer labels in CLASS_ORDER
class_names = list(CLASS_ORDER)  # 3.12.26 fixed class-name order for reports
n_classes = len(class_names)  # 3.12.26 number of classes used by the CNN
all_idx = np.arange(N, dtype=np.int64)  # 3.12.26 all dataset indices in row order

avail_counts = meta["hue"].astype(str).value_counts().reindex(CLASS_ORDER).fillna(0).astype(int)  # 3.12.26 available count for each hue
assert (avail_counts > 0).all(), f"Every class must be present to make balanced splits. Counts: {avail_counts.to_dict()}"  # 3.12.26 require at least one sample from every hue

target_weights = pd.Series(TARGET_CLASS_WEIGHTS).reindex(CLASS_ORDER).astype(float)  # 3.14.26 use the intended class weights defined in Cell 1
assert target_weights.notna().all(), f"Missing TARGET_CLASS_WEIGHTS entries for: {target_weights[target_weights.isna()].index.tolist()}"  # 3.14.26 require one target weight per class

min_weight = float(target_weights[target_weights > 0].min())  # 3.14.26 smallest positive weight defines one unit of the target ratio
weight_units = (target_weights / min_weight).round().astype(int)  # 3.14.26 convert 25/25/12.5/... into integer ratio units 2/2/1/1/1/1
assert np.allclose(target_weights / min_weight, weight_units.astype(float)), "TARGET_CLASS_WEIGHTS must reduce cleanly to integer ratio units"  # 3.14.26 require exact ratio-style weights

pool_unit = int(min(avail_counts[hue] // weight_units[hue] for hue in CLASS_ORDER))  # 3.14.26 largest exact shared unit that fits all classes
target_pool_counts = pd.Series(
    {hue: int(weight_units[hue] * pool_unit) for hue in CLASS_ORDER},
    index=CLASS_ORDER,
    dtype=int,
)  # 3.14.26 exact class counts for the weighted balanced pool

pool_parts = []  # 3.14.26 collect the weighted index subset for each hue
for hue in CLASS_ORDER:  # 3.14.26 build the pool using the requested class proportions
    idx_h = all_idx[y_str == hue].copy()  # 3.14.26 dataset indices belonging to this hue
    rng.shuffle(idx_h)  # 3.14.26 shuffle before taking the weighted subset
    pool_parts.append(idx_h[:int(target_pool_counts[hue])])  # 3.14.26 keep the exact target count for this hue

pool_idx = np.concatenate(pool_parts).astype(np.int64)  # 3.14.26 final weighted balanced pool used for splitting
rng.shuffle(pool_idx)  # 3.14.26 mix hues and conditions in the pooled index list
N_POOL = int(pool_idx.size)  # 3.14.26 total size of the weighted balanced pool

N_INTENSITY_BINS = 3  # 3.12.26 number of object-intensity bins used for balancing
N_SATURATION_BINS = 3  # 3.12.26 number of object-saturation bins used for balancing
N_BG_INTENSITY_BINS = 3  # 3.12.26 number of background-intensity bins used for balancing
N_BG_SATURATION_BINS = 3  # 3.12.26 number of background-saturation bins used for balancing

def make_bin_strings(series: pd.Series, n_bins: int, default_label: str) -> pd.Series:  # 3.12.26 simple helper to make safe string bins
    series = pd.to_numeric(series, errors="coerce")  # 3.12.26 force numeric input before binning
    n_unique = int(series.nunique(dropna=True))  # 3.12.26 count unique numeric values
    if n_unique <= 1:  # 3.12.26 if values do not vary, use one constant bin label
        return pd.Series([default_label] * len(series), index=series.index, dtype="object")  # 3.12.26 return one simple bin for all rows
    return pd.qcut(series, q=min(int(n_bins), n_unique), duplicates="drop").astype(str)  # 3.12.26 make quantile bins and store them as strings

meta["intensity_bin"] = make_bin_strings(meta["intensity"], N_INTENSITY_BINS, "intensity_single")  # 3.12.26 object-intensity bins
meta["saturation_bin"] = "gray_sat_0"  # 3.12.26 give grayscale stimuli one explicit saturation bin instead of mixing them into qcut
color_mask = meta["is_color"].to_numpy(dtype=bool)  # 3.12.26 identify color stimuli so their saturation can be binned separately
color_saturation = pd.to_numeric(meta.loc[color_mask, "saturation"], errors="coerce")  # 3.12.26 pull color-only saturation values
color_n_unique = int(color_saturation.nunique(dropna=True))  # 3.12.26 count unique color-saturation values only
assert color_n_unique > 0, "Color stimuli must have at least one valid saturation value"  # 3.12.26 fail early if color saturation is missing
if color_n_unique == 1:  # 3.12.26 if all color stimuli share one saturation, keep one simple color bin
    meta.loc[color_mask, "saturation_bin"] = "color_sat_single"  # 3.12.26 single saturation bin for all color stimuli
else:  # 3.12.26 otherwise bin color saturation values without the gray controls collapsing qcut
    meta.loc[color_mask, "saturation_bin"] = pd.qcut(  # 3.12.26 quantile bins for color stimuli only
        color_saturation,
        q=min(int(N_SATURATION_BINS), color_n_unique),
        duplicates="drop",
    ).astype(str).to_numpy()  # 3.12.26 store the color-only saturation bins as strings
meta["bg_intensity_bin"] = make_bin_strings(meta["bg_intensity"], N_BG_INTENSITY_BINS, "bg_intensity_single")  # 3.12.26 background-intensity bins
meta["bg_saturation_bin"] = make_bin_strings(meta["bg_saturation"], N_BG_SATURATION_BINS, "bg_saturation_single")  # 3.12.26 background-saturation bins
meta["bg_hue_bin"] = meta["bg_hue"].astype(str)  # 3.12.26 background hue is already categorical, so use it directly

meta["balance_bin"] = (  # 3.12.26 combined condition key used for balancing within each hue
    meta["intensity_bin"].astype(str) + "|" +
    meta["saturation_bin"].astype(str) + "|" +
    meta["bg_hue_bin"].astype(str) + "|" +
    meta["bg_intensity_bin"].astype(str) + "|" +
    meta["bg_saturation_bin"].astype(str)
).astype(str)  # 3.12.26 keep the balance key as a simple stable string

meta["bin_id"] = (  # 3.12.26 full bin key including hue, useful for summaries and saved records
    meta["hue"].astype(str) + "|" + meta["balance_bin"].astype(str)
).astype(str)  # 3.12.26 keep bin_id as a simple stable string

TRAIN_FRAC = 0.70  # 3.12.26 fraction of the balanced pool assigned to train
VAL_FRAC = 0.15  # 3.12.26 fraction of the balanced pool assigned to validation
TEST_FRAC = 0.15  # 3.12.26 fraction of the balanced pool assigned to test
assert abs((TRAIN_FRAC + VAL_FRAC + TEST_FRAC) - 1.0) < 1e-9, "Split fractions must sum to 1.0"  # 3.12.26 safety check

def alloc3(n: int, train_frac: float, val_frac: float, test_frac: float):  # 3.12.26 allocate exact integer counts that sum to n
    quotas = np.array([n * train_frac, n * val_frac, n * test_frac], dtype=float)  # 3.12.26 ideal non-integer split sizes
    base = np.floor(quotas).astype(int)  # 3.12.26 start with floor counts
    need = int(n - base.sum())  # 3.12.26 number of leftover samples still needing assignment
    if need > 0:  # 3.12.26 distribute leftovers to the largest fractional remainders
        order = np.argsort(-(quotas - base))  # 3.12.26 largest fractional parts first
        for pos in order[:need]:  # 3.12.26 give one extra sample to the first needed positions
            base[pos] += 1  # 3.12.26 update integer allocation
    return int(base[0]), int(base[1]), int(base[2])  # 3.12.26 return train, val, and test counts

split_names = ["train", "val", "test"]  # 3.12.26 fixed split order used throughout this cell
split_fracs = {"train": TRAIN_FRAC, "val": VAL_FRAC, "test": TEST_FRAC}  # 3.12.26 fractions saved in dictionary form

train_parts = []  # 3.12.26 collect per-hue train indices
val_parts = []  # 3.12.26 collect per-hue validation indices
test_parts = []  # 3.12.26 collect per-hue test indices
bin_to_alloc = {}  # 3.12.26 save per-bin allocation details for reproducibility
hue_target_counts = {}  # 3.12.26 save the exact target split sizes for each hue

for hue in CLASS_ORDER:  # 3.12.26 split each hue separately so hue counts stay perfectly balanced
    idx_h = pool_idx[y_str[pool_idx] == hue].astype(np.int64)  # 3.12.26 balanced pool indices for this hue only
    n_train_h, n_val_h, n_test_h = alloc3(len(idx_h), TRAIN_FRAC, VAL_FRAC, TEST_FRAC)  # 3.12.26 exact split targets for this hue
    hue_target_counts[hue] = {"train": int(n_train_h), "val": int(n_val_h), "test": int(n_test_h)}  # 3.12.26 save per-hue targets

    assigned = {"train": [], "val": [], "test": []}  # 3.12.26 indices assigned so far for this hue
    bin_labels_h = meta.loc[idx_h, "balance_bin"].astype(str).to_numpy()  # 3.12.26 balance-bin labels for this hue

    bin_sizes = pd.Series(bin_labels_h).value_counts()  # 3.12.26 count how many pooled samples fall in each balance bin
    ordered_bins = sorted(bin_sizes.index.tolist(), key=lambda b: (int(bin_sizes.loc[b]), str(b)))  # 3.12.26 process rare bins first, then larger bins

    bin_to_indices = {}  # 03.14.2026 store shuffled indices once per bin
    per_bin_counts = {}  # 03.14.2026 store base split counts for each hue-bin
    per_bin_frac_parts = {}  # 03.14.2026 store fractional remainders for leftover assignment
    base_totals = {"train": 0, "val": 0, "test": 0}  # 03.14.2026 total base counts before leftover distribution

    for bin_name in ordered_bins:  # 03.14.2026 first pass: make per-bin proportional base allocations
        idx_bin = idx_h[bin_labels_h == bin_name].astype(np.int64)  # 03.14.2026 pooled indices in this hue and this balance bin
        rng.shuffle(idx_bin)  # 03.14.2026 shuffle samples within the bin before final slicing
        bin_to_indices[bin_name] = idx_bin  # 03.14.2026 save shuffled bin indices for second pass

        quotas = np.array([len(idx_bin) * TRAIN_FRAC, len(idx_bin) * VAL_FRAC, len(idx_bin) * TEST_FRAC], dtype=float)  # 03.14.2026 ideal per-bin split quotas
        base = np.floor(quotas).astype(int)  # 03.14.2026 base per-bin counts from floor allocation

        per_bin_counts[bin_name] = {"train": int(base[0]), "val": int(base[1]), "test": int(base[2])}  # 03.14.2026 save base counts for this bin
        per_bin_frac_parts[bin_name] = {  # 03.14.2026 save fractional parts to place leftovers teachably
            "train": float(quotas[0] - base[0]),  # 03.14.2026 train fractional remainder
            "val": float(quotas[1] - base[1]),  # 03.14.2026 val fractional remainder
            "test": float(quotas[2] - base[2]),  # 03.14.2026 test fractional remainder
        }  # 03.14.2026 end fractional remainder record
        base_totals["train"] += int(base[0])  # 03.14.2026 accumulate base train count across bins
        base_totals["val"] += int(base[1])  # 03.14.2026 accumulate base val count across bins
        base_totals["test"] += int(base[2])  # 03.14.2026 accumulate base test count across bins

    leftover_targets = {  # 03.14.2026 exact extra counts each split still needs after base allocations
        "train": int(n_train_h - base_totals["train"]),  # 03.14.2026 train leftovers still needed
        "val": int(n_val_h - base_totals["val"]),  # 03.14.2026 val leftovers still needed
        "test": int(n_test_h - base_totals["test"]),  # 03.14.2026 test leftovers still needed
    }  # 03.14.2026 end leftover-target dictionary
    assert all(v >= 0 for v in leftover_targets.values()), f"Negative leftover target for hue {hue}: {leftover_targets}"  # 03.14.2026 require valid leftover targets

    for bin_name in ordered_bins:  # 03.14.2026 second pass: place bin leftovers while preserving exact hue totals
        idx_bin = bin_to_indices[bin_name]  # 03.14.2026 recover shuffled indices for this bin
        alloc_counts = dict(per_bin_counts[bin_name])  # 03.14.2026 start from the base per-bin counts
        leftovers = int(len(idx_bin) - sum(alloc_counts.values()))  # 03.14.2026 number of unassigned samples left in this bin

        priority = sorted(  # 03.14.2026 prefer largest fractional remainder, then largest remaining split need
            split_names,
            key=lambda s: (-per_bin_frac_parts[bin_name][s], -leftover_targets[s], -split_fracs[s], s),
        )  # 03.14.2026 stable priority order for leftover assignment
        for split_name in priority:  # 03.14.2026 assign leftovers one by one within this bin
            if leftovers == 0:  # 03.14.2026 stop once this bin is fully assigned
                break  # 03.14.2026 no more leftovers in this bin
            if leftover_targets[split_name] > 0:  # 03.14.2026 only give extras to splits that still need them
                alloc_counts[split_name] += 1  # 03.14.2026 add one leftover sample to this split
                leftover_targets[split_name] -= 1  # 03.14.2026 reduce the remaining need for this split
                leftovers -= 1  # 03.14.2026 reduce leftover count for this bin

        if leftovers > 0:  # 03.14.2026 fallback if any leftovers remain after remainder-based priority
            fallback_priority = sorted(  # 03.14.2026 use remaining split need as the primary fallback rule
                split_names,
                key=lambda s: (-leftover_targets[s], -per_bin_frac_parts[bin_name][s], -split_fracs[s], s),
            )  # 03.14.2026 stable fallback order
            for split_name in fallback_priority:  # 03.14.2026 continue assigning any remaining leftovers
                if leftovers == 0:  # 03.14.2026 stop once this bin is fully assigned
                    break  # 03.14.2026 no more leftovers in this bin
                if leftover_targets[split_name] > 0:  # 03.14.2026 only give extras to needed splits
                    alloc_counts[split_name] += 1  # 03.14.2026 add one leftover sample to this split
                    leftover_targets[split_name] -= 1  # 03.14.2026 reduce the remaining need for this split
                    leftovers -= 1  # 03.14.2026 reduce leftover count for this bin

        assert leftovers == 0, f"Could not place all leftovers for hue {hue}, bin {bin_name}"  # 03.14.2026 require full per-bin assignment

        start = 0  # 03.14.2026 running pointer into the shuffled indices for this bin
        for split_name in split_names:  # 03.14.2026 slice this bin into train, val, and test chunks
            n_take = int(alloc_counts[split_name])  # 03.14.2026 number of samples to take for this split
            assigned[split_name].extend(idx_bin[start:start + n_take].tolist())  # 03.14.2026 append this split's slice from the bin
            start += n_take  # 03.14.2026 advance to the next slice boundary

        bin_to_alloc[f"{hue}|{bin_name}"] = {  # 3.12.26 save how this hue-specific bin was allocated
            "n": int(len(idx_bin)),  # 3.12.26 number of pooled samples in this bin
            "n_train": int(alloc_counts["train"]),  # 3.12.26 number sent to train
            "n_val": int(alloc_counts["val"]),  # 3.12.26 number sent to validation
            "n_test": int(alloc_counts["test"]),  # 3.12.26 number sent to test
        }  # 3.12.26 end allocation record for this bin

    assert all(v == 0 for v in leftover_targets.values()), f"Leftover targets not exhausted for hue {hue}: {leftover_targets}"  # 03.14.2026 require exact hue totals after bin allocation
    assert len(assigned["train"]) == int(n_train_h), f"Train count mismatch for hue {hue}"  # 3.12.26 require exact hue-balanced train size
    assert len(assigned["val"]) == int(n_val_h), f"Val count mismatch for hue {hue}"  # 3.12.26 require exact hue-balanced val size
    assert len(assigned["test"]) == int(n_test_h), f"Test count mismatch for hue {hue}"  # 3.12.26 require exact hue-balanced test size

    train_parts.append(np.array(assigned["train"], dtype=np.int64))  # 3.12.26 store this hue's train indices
    val_parts.append(np.array(assigned["val"], dtype=np.int64))  # 3.12.26 store this hue's validation indices
    test_parts.append(np.array(assigned["test"], dtype=np.int64))  # 3.12.26 store this hue's test indices

train_idx = np.concatenate(train_parts).astype(np.int64)  # 3.12.26 final train indices across all hues
val_idx = np.concatenate(val_parts).astype(np.int64)  # 3.12.26 final validation indices across all hues
test_idx = np.concatenate(test_parts).astype(np.int64)  # 3.12.26 final test indices across all hues

rng.shuffle(train_idx)  # 3.12.26 shuffle final train index order
rng.shuffle(val_idx)  # 3.12.26 shuffle final validation index order
rng.shuffle(test_idx)  # 3.12.26 shuffle final test index order

assert len(set(train_idx) & set(val_idx)) == 0, "Train and val splits overlap"  # 3.12.26 require non-overlapping splits
assert len(set(train_idx) & set(test_idx)) == 0, "Train and test splits overlap"  # 3.12.26 require non-overlapping splits
assert len(set(val_idx) & set(test_idx)) == 0, "Val and test splits overlap"  # 3.12.26 require non-overlapping splits
assert int(len(train_idx) + len(val_idx) + len(test_idx)) == int(N_POOL), "Split sizes must sum to the balanced pool size"  # 3.12.26 require full coverage of the pooled dataset

def counts_by_hue(idxs):  # 3.12.26 simple helper for per-split hue summaries
    return meta.loc[idxs, "hue"].astype(str).value_counts().reindex(CLASS_ORDER).fillna(0).astype(int)  # 3.12.26 return class counts in fixed order

def counts_by_col(idxs, col):  # 3.12.26 simple helper for per-split bin/background summaries
    return meta.loc[idxs, col].astype(str).value_counts().sort_index()  # 3.12.26 return sorted counts for one metadata column

SPLIT_RECORD = {  # 3.12.26 save split settings and outputs for reproducibility
    "split_seed": int(SPLIT_SEED),  # 3.12.26 seed used for all split operations
    "split_fracs": {"train": float(TRAIN_FRAC), "val": float(VAL_FRAC), "test": float(TEST_FRAC)},  # 3.12.26 requested split fractions
    "balance_method": "weighted_hue_pool_then_exact_per_bin_proportional_split_by_intensity_color_only_saturation_and_background_bins",  # 03.14.2026 update method name for the new allocation rule
    "binning": {  # 3.12.26 record the number of bins used for balancing
        "n_intensity_bins": int(N_INTENSITY_BINS),  # 3.12.26 object-intensity bin count
        "n_saturation_bins": int(N_SATURATION_BINS),  # 3.12.26 object-saturation bin count
        "n_bg_intensity_bins": int(N_BG_INTENSITY_BINS),  # 3.12.26 background-intensity bin count
        "n_bg_saturation_bins": int(N_BG_SATURATION_BINS),  # 3.12.26 background-saturation bin count
    },  # 3.12.26 end binning record
    "avail_counts": avail_counts.to_dict(),  # 3.12.26 class counts in the full dataset
    "target_class_weights": dict(TARGET_CLASS_WEIGHTS),  # 3.14.26 requested class weights from Cell 1
    "target_pool_counts": target_pool_counts.to_dict(),  # 3.14.26 exact per-class counts used in the pooled dataset
    "pool_counts": counts_by_hue(pool_idx).to_dict(),  # 3.12.26 class counts in the balanced pool
    "hue_target_counts": hue_target_counts,  # 3.12.26 exact target split sizes for every hue
    "bin_allocations": bin_to_alloc,  # 3.12.26 per-bin allocation record
    "class_order": list(CLASS_ORDER),  # 3.12.26 fixed class ordering
    "train_idx": train_idx,  # 3.12.26 final train indices
    "val_idx": val_idx,  # 3.12.26 final validation indices
    "test_idx": test_idx,  # 3.12.26 final test indices
}  # 3.12.26 end split record

print("Fixed label mapping (int -> hue):")  # 3.12.26 print the fixed class mapping
for i, name in enumerate(class_names):  # 3.12.26 loop over class names in order
    print(f"  {i} -> {name}")  # 3.12.26 print one mapping line

print("\nAvailable counts in full dataset:\n", avail_counts)  # 3.12.26 show original class counts before balancing
print("\nBalanced pool size:", N_POOL)  # 3.12.26 show total size of the equal-hue pool
print("Balanced pool counts:\n", counts_by_hue(pool_idx))  # 3.12.26 confirm equal hue counts in the pool

print("\nSplit sizes:")  # 3.12.26 print total sizes of the three splits
print("  Train:", len(train_idx))  # 3.12.26 train size
print("  Val  :", len(val_idx))  # 3.12.26 validation size
print("  Test :", len(test_idx))  # 3.12.26 test size

print("\nClass counts per split:")  # 3.12.26 print hue balance in each split
print("  Train:\n", counts_by_hue(train_idx))  # 3.12.26 train hue counts
print("  Val:\n", counts_by_hue(val_idx))  # 3.12.26 validation hue counts
print("  Test:\n", counts_by_hue(test_idx))  # 3.12.26 test hue counts

print("\nUnique bin_id counts per split:")  # 3.12.26 print how many hue+condition bins appear in each split
print("  Train bins:", int(meta.loc[train_idx, "bin_id"].nunique()))  # 3.12.26 train unique bin count
print("  Val bins  :", int(meta.loc[val_idx, "bin_id"].nunique()))  # 3.12.26 validation unique bin count
print("  Test bins :", int(meta.loc[test_idx, "bin_id"].nunique()))  # 3.12.26 test unique bin count

print("\nIntensity bin counts per split:")  # 3.12.26 print object-intensity balance
print("  Train:\n", counts_by_col(train_idx, "intensity_bin"))  # 3.12.26 train intensity-bin counts
print("  Val:\n", counts_by_col(val_idx, "intensity_bin"))  # 3.12.26 validation intensity-bin counts
print("  Test:\n", counts_by_col(test_idx, "intensity_bin"))  # 3.12.26 test intensity-bin counts

print("\nSaturation bin counts per split:")  # 3.12.26 print object-saturation balance
print("  Train:\n", counts_by_col(train_idx, "saturation_bin"))  # 3.12.26 train saturation-bin counts
print("  Val:\n", counts_by_col(val_idx, "saturation_bin"))  # 3.12.26 validation saturation-bin counts
print("  Test:\n", counts_by_col(test_idx, "saturation_bin"))  # 3.12.26 test saturation-bin counts

print("\nBackground hue counts per split:")  # 3.12.26 print background-hue balance
print("  Train:\n", counts_by_col(train_idx, "bg_hue_bin"))  # 3.12.26 train background-hue counts
print("  Val:\n", counts_by_col(val_idx, "bg_hue_bin"))  # 3.12.26 validation background-hue counts
print("  Test:\n", counts_by_col(test_idx, "bg_hue_bin"))  # 3.12.26 test background-hue counts

print("\nBackground intensity bin counts per split:")  # 3.12.26 print background-intensity balance
print("  Train:\n", counts_by_col(train_idx, "bg_intensity_bin"))  # 3.12.26 train background-intensity-bin counts
print("  Val:\n", counts_by_col(val_idx, "bg_intensity_bin"))  # 3.12.26 validation background-intensity-bin counts
print("  Test:\n", counts_by_col(test_idx, "bg_intensity_bin"))  # 3.12.26 test background-intensity-bin counts

print("\nBackground saturation bin counts per split:")  # 3.12.26 print background-saturation balance
print("  Train:\n", counts_by_col(train_idx, "bg_saturation_bin"))  # 3.12.26 train background-saturation-bin counts
print("  Val:\n", counts_by_col(val_idx, "bg_saturation_bin"))  # 3.12.26 validation background-saturation-bin counts
print("  Test:\n", counts_by_col(test_idx, "bg_saturation_bin"))  # 3.12.26 test background-saturation-bin counts

In [ ]:
# Cell 4 — Define Dataset + DataLoaders

CHANNEL_SET = BUILD_GROUPS[-1]

if CHANNEL_SET not in CHANNEL_GROUPS:  # 3.12.26 make sure the selected channel group name is valid
    raise ValueError(f"CHANNEL_SET must be one of {list(CHANNEL_GROUPS.keys())}")  # 3.12.26 stop early with a clear error if the group name is invalid

keys_subset = CHANNEL_GROUPS[CHANNEL_SET]  # 3.12.26 get the response-channel names for the selected channel group
key_to_idx = {k: i for i, k in enumerate(keys_order)}  # 3.12.26 map each saved response key to its channel index
missing_keys = [k for k in keys_subset if k not in key_to_idx]  # 3.12.26 detect any requested channels missing from the dataset
assert len(missing_keys) == 0, f"Missing channels in keys_order: {missing_keys}"  # 3.12.26 fail early if any requested channel is absent

channel_idxs = np.array([key_to_idx[k] for k in keys_subset], dtype=np.int64)  # 3.12.26 integer channel indices for the selected channel group
assert len(np.unique(channel_idxs)) == len(channel_idxs), "Duplicate channel indices detected"  # 3.12.26 guard against duplicate channel selection
assert channel_idxs.min() >= 0 and channel_idxs.max() < int(C), "channel_idxs are out of range"  # 3.12.26 make sure selected channels exist in outs_fill
C_IN = int(len(channel_idxs))  # 3.12.26 number of CNN input channels for this run

print("Using CHANNEL_SET:", CHANNEL_SET)
print("Channels (keys):", keys_subset)
print("Channels (idxs):", channel_idxs.tolist())  # 3.12.26 print a regular Python list for readability
print("C_IN =", C_IN)

RUN_ARTIFACTS_DIR = ARTIFACTS_DIR / f"{CHANNEL_SET}_split{int(SPLIT_SEED)}"  # 3.12.26 make one artifact folder for this channel group and split seed
RUN_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the artifact folder exists before later save steps

for split_name, split_idx in [("train_idx", train_idx), ("val_idx", val_idx), ("test_idx", test_idx)]:  # 3.12.26 check each split index array
    split_idx = np.asarray(split_idx, dtype=np.int64)  # 3.12.26 force a clean integer array for checks
    assert split_idx.size > 0, f"{split_name} is empty"  # 3.12.26 require every split to contain at least one sample
    assert split_idx.min() >= 0 and split_idx.max() < int(N), f"{split_name} must contain indices in [0, N)"  # 3.12.26 require every split index to be in range
    assert np.unique(split_idx).size == split_idx.size, f"{split_name} contains duplicate indices"  # 3.12.26 require no duplicate samples within a split

class RetinaHueDataset(Dataset):  # 3.12.26 simplified dataset class for lazy loading from Zarr
    def __init__(self, indices, labels, channel_idxs):  # 3.12.26 store only what is needed to fetch one sample
        self.indices = np.asarray(indices, dtype=np.int64)  # 3.12.26 stimulus indices for this split
        self.labels = np.asarray(labels, dtype=np.int64)  # 3.12.26 full label array aligned to dataset order
        self.channel_idxs = np.asarray(channel_idxs, dtype=np.int64)  # 3.12.26 selected response-channel indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        stim_id = int(self.indices[i])  # 3.12.26 global stimulus index for this sample
        x_np = get_outs_fill_sample(stim_id)[self.channel_idxs, :, :]  # 3.12.26 lazily load only the selected response channels
        x_np = np.ascontiguousarray(x_np, dtype=np.float32)  # 3.12.26 make the array contiguous float32 before converting to torch
        x = torch.from_numpy(x_np)  # 3.12.26 convert the response map stack to a torch tensor
        y_t = torch.tensor(int(self.labels[stim_id]), dtype=torch.long)  # 3.12.26 get the class label for this stimulus
        return x, y_t

ds_train = RetinaHueDataset(train_idx, y, channel_idxs)  # 3.12.26 training dataset
ds_val = RetinaHueDataset(val_idx, y, channel_idxs)  # 3.12.26 validation dataset
ds_test = RetinaHueDataset(test_idx, y, channel_idxs)  # 3.12.26 test dataset

BATCH_SIZE = 5
NUM_WORKERS = 0
PIN_MEMORY = (device.type == "cuda")  # 3.12.26 use pinned memory only when running on a GPU

dl_train_generator = make_torch_generator(int(SHUFFLE_SEED_BASE))  # 3.12.26 deterministic shuffle generator for the training loader

dl_train = DataLoader(
    ds_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,  # 3.12.26 pin memory only when helpful
    generator=dl_train_generator,  # 3.12.26 deterministic training shuffle order
    worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
)

dl_val = DataLoader(
    ds_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,  # 3.12.26 keep DataLoader settings consistent across splits
    worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
)

dl_test = DataLoader(
    ds_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,  # 3.12.26 keep DataLoader settings consistent across splits
    worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
)

xb, yb = next(iter(dl_train))
print("Batch x shape:", tuple(xb.shape))
print("Batch y shape:", tuple(yb.shape))
print("x dtype:", xb.dtype, " y dtype:", yb.dtype)

In [ ]:
# Cell 5 — Define the CNN model (logits + CrossEntropyLoss)

RUN_INIT_SEED = int(INIT_SEED_BASE)  # 3.12.26 use the initialization seed that was defined in Cell 1

class RetinaCNN(nn.Module):
    def __init__(self, in_channels, n_classes=6):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels=in_channels,  # 3.12.26 the number of input channels now comes directly from the selected channel group
            out_channels=16,
            kernel_size=3,  # 3.12.26 simplify by fixing the first convolution settings inside the model
            stride=2,  # 3.12.26 simplify by fixing the first convolution settings inside the model
            padding=1,  # 3.12.26 simplify by fixing the first convolution settings inside the model
        )
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)

        self.fc1 = nn.LazyLinear(256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, n_classes)  # 3.12.26 final layer outputs one logit per class in the new 6-class system

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.conv3(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.conv4(x)
        x = self.relu(x)
        x = self.pool(x)

        x = torch.flatten(x, start_dim=1)

        x = self.fc1(x)
        x = self.relu(x)

        latent = self.fc2(x)
        latent = self.relu(latent)

        logits = self.fc3(latent)
        return logits, latent

def build_seeded_model(in_channels, n_classes, init_seed, device, example_batch):
    rng_state = get_torch_rng_state()  # 3.12.26 save the current torch RNG state so model building does not disturb later randomness
    torch.manual_seed(int(init_seed))  # 3.12.26 seed model initialization reproducibly
    if torch.cuda.is_available():  # 3.12.26 seed CUDA too when a GPU is available
        torch.cuda.manual_seed_all(int(init_seed))  # 3.12.26 seed all CUDA devices reproducibly

    model = RetinaCNN(in_channels=in_channels, n_classes=n_classes)  # 3.12.26 build the CNN using the selected input-channel count and class count
    model = model.to(device)  # 3.12.26 move the model to the chosen device

    model.eval()  # 3.12.26 use eval mode for the one dry-run forward pass that initializes LazyLinear
    with torch.no_grad():  # 3.12.26 do not track gradients during the dry-run forward pass
        _ = model(example_batch.to(device))  # 3.12.26 run one example batch through the model to initialize the lazy layer
    model.train()  # 3.12.26 return the model to training mode for later cells

    set_torch_rng_state(rng_state)  # 3.12.26 restore the RNG state after seeded model creation
    return model

x0, _ = ds_train[0]  # 3.12.26 only the input tensor is needed here for the model dry-run
xb = x0.unsqueeze(0)

C_IN_EFFECTIVE = int(xb.shape[1])  # 3.12.26 confirm the true number of channels from an actual dataset sample
assert C_IN_EFFECTIVE == int(C_IN), f"Mismatch: xb has C={C_IN_EFFECTIVE} but C_IN={C_IN}"  # 3.12.26 fail early if dataset channels and selected channels disagree

model = build_seeded_model(
    in_channels=C_IN_EFFECTIVE,  # 3.12.26 use the actual input-channel count from the dataset sample
    n_classes=n_classes,  # 3.12.26 use the 6-class count from Cell 3
    init_seed=RUN_INIT_SEED,  # 3.12.26 use the reproducible model-initialization seed
    device=device,
    example_batch=xb,  # 3.12.26 provide one real sample so LazyLinear can infer its input size
)

model.eval()  # 3.12.26 use eval mode for the sanity-check forward pass
with torch.no_grad():
    logits, latent = model(xb.to(device))
model.train()  # 3.12.26 return the model to training mode for later cells

print("logits shape:", tuple(logits.shape))
print("latent shape:", tuple(latent.shape))

In [ ]:
# Cell 6 — Training utilities (train/eval loops + history)  # 3.12.26 simplified: no backup imports, no standalone fallbacks, one main training utility cell

USE_CLASS_WEIGHTS = False  # 3.12.26 keep class weighting off by default because Cell 3 already builds balanced hue splits
LR = 1e-4  # 3.12.26 default learning rate used by make_optimizer
USE_AMP = (device.type == "cuda")  # 3.12.26 use mixed precision only on GPU
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)  # 3.12.26 gradient scaler for mixed-precision training on CUDA

def make_class_weights_from_counts(counts, device):  # 3.12.26 helper to build inverse-frequency class weights from split counts
    counts = np.asarray(counts, dtype=np.float32)  # 3.12.26 convert counts to float for stable division
    weights = counts.sum() / np.maximum(counts, 1.0)  # 3.12.26 inverse-frequency weighting with divide-by-zero protection
    weights = weights / np.maximum(weights.mean(), 1e-12)  # 3.12.26 normalize weights so the mean stays near 1
    return torch.tensor(weights, dtype=torch.float32, device=device)  # 3.12.26 return weights on the active device

def make_criterion(y_int, idxs, n_classes, device, use_class_weights=USE_CLASS_WEIGHTS):  # 3.12.26 build the loss function for one split
    if bool(use_class_weights):  # 3.12.26 optionally compute split-specific class weights
        split_labels = np.asarray(y_int, dtype=np.int64)[np.asarray(idxs, dtype=np.int64)]  # 3.12.26 labels present in this split only
        counts = np.bincount(split_labels, minlength=int(n_classes))  # 3.12.26 class counts for this split
        class_weights_t = make_class_weights_from_counts(counts, device=device)  # 3.12.26 convert counts to a weight tensor
    else:  # 3.12.26 usual case for this notebook
        class_weights_t = None  # 3.12.26 no class weighting when splits are already balanced
    return nn.CrossEntropyLoss(weight=class_weights_t)  # 3.12.26 CrossEntropyLoss expects logits and integer class labels

def make_optimizer(model, lr=LR):  # 3.12.26 simple optimizer helper
    return torch.optim.Adam(model.parameters(), lr=lr)  # 3.12.26 Adam optimizer with the chosen learning rate

def _accuracy_from_logits(logits, y_true):  # 3.12.26 compute batch accuracy from model logits
    preds = torch.argmax(logits, dim=1)  # 3.12.26 predicted class index for each sample
    correct = (preds == y_true).sum().item()  # 3.12.26 number of correct predictions
    total = int(y_true.numel())  # 3.12.26 number of labels in the batch
    return correct, total  # 3.12.26 return numerator and denominator for accuracy

def train_one_epoch(  # 3.12.26 train for one full pass through the training DataLoader
    model,
    dataloader,
    optimizer,
    criterion,
    device=device,
    use_amp=USE_AMP,
    scaler=scaler,
    grad_clip=None,
):
    model.train()  # 3.12.26 put the model in training mode
    running_loss = 0.0  # 3.12.26 accumulate total loss across the epoch
    running_correct = 0  # 3.12.26 accumulate total correct predictions
    running_total = 0  # 3.12.26 accumulate total number of samples
    non_block = (device.type == "cuda")  # 3.12.26 use non_blocking transfers only when running on GPU

    for xb, yb in dataloader:  # 3.12.26 iterate over training batches
        xb = xb.to(device=device, dtype=torch.float32, non_blocking=non_block)  # 3.12.26 move inputs to the active device
        yb = yb.to(device=device, dtype=torch.long, non_blocking=non_block)  # 3.12.26 move labels to the active device
        optimizer.zero_grad(set_to_none=True)  # 3.12.26 clear old gradients before the next update

        if use_amp:  # 3.12.26 mixed-precision training branch for CUDA
            with torch.autocast(device_type=device.type, enabled=True):  # 3.12.26 autocast forward pass on GPU
                logits, _ = model(xb)  # 3.12.26 run the model and ignore the latent output here
                loss = criterion(logits, yb)  # 3.12.26 compute the training loss
            scaler.scale(loss).backward()  # 3.12.26 backpropagate the scaled loss
            if grad_clip is not None:  # 3.12.26 optionally clip gradients before the optimizer step
                scaler.unscale_(optimizer)  # 3.12.26 unscale gradients before clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), float(grad_clip))  # 3.12.26 clip gradients to the chosen threshold
            scaler.step(optimizer)  # 3.12.26 apply one optimizer step
            scaler.update()  # 3.12.26 update the gradient scaler state
        else:  # 3.12.26 standard full-precision training branch
            logits, _ = model(xb)  # 3.12.26 run the model and ignore the latent output here
            loss = criterion(logits, yb)  # 3.12.26 compute the training loss
            loss.backward()  # 3.12.26 backpropagate gradients
            if grad_clip is not None:  # 3.12.26 optionally clip gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), float(grad_clip))  # 3.12.26 clip gradients to the chosen threshold
            optimizer.step()  # 3.12.26 apply one optimizer step

        batch_size = int(yb.size(0))  # 3.12.26 number of samples in this batch
        running_loss += float(loss.item()) * batch_size  # 3.12.26 accumulate total loss weighted by batch size
        correct, total = _accuracy_from_logits(logits, yb)  # 3.12.26 compute batch accuracy counts
        running_correct += int(correct)  # 3.12.26 accumulate number of correct predictions
        running_total += int(total)  # 3.12.26 accumulate number of seen samples

    avg_loss = running_loss / max(1, running_total)  # 3.12.26 average training loss across the epoch
    avg_acc = running_correct / max(1, running_total)  # 3.12.26 average training accuracy across the epoch
    return avg_loss, avg_acc  # 3.12.26 return epoch training metrics

@torch.no_grad()  # 3.12.26 evaluation does not need gradient tracking
def eval_one_epoch(  # 3.12.26 evaluate for one full pass through a validation or test DataLoader
    model,
    dataloader,
    criterion,
    device=device,
    use_amp=USE_AMP,
):
    model.eval()  # 3.12.26 put the model in evaluation mode
    running_loss = 0.0  # 3.12.26 accumulate total loss across evaluation
    running_correct = 0  # 3.12.26 accumulate total correct predictions
    running_total = 0  # 3.12.26 accumulate total number of samples
    non_block = (device.type == "cuda")  # 3.12.26 use non_blocking transfers only when running on GPU

    for xb, yb in dataloader:  # 3.12.26 iterate over evaluation batches
        xb = xb.to(device=device, dtype=torch.float32, non_blocking=non_block)  # 3.12.26 move inputs to the active device
        yb = yb.to(device=device, dtype=torch.long, non_blocking=non_block)  # 3.12.26 move labels to the active device

        if use_amp:  # 3.12.26 mixed-precision evaluation branch for CUDA
            with torch.autocast(device_type=device.type, enabled=True):  # 3.12.26 autocast forward pass on GPU
                logits, _ = model(xb)  # 3.12.26 run the model and ignore the latent output here
                loss = criterion(logits, yb)  # 3.12.26 compute the evaluation loss
        else:  # 3.12.26 standard full-precision evaluation branch
            logits, _ = model(xb)  # 3.12.26 run the model and ignore the latent output here
            loss = criterion(logits, yb)  # 3.12.26 compute the evaluation loss

        batch_size = int(yb.size(0))  # 3.12.26 number of samples in this batch
        running_loss += float(loss.item()) * batch_size  # 3.12.26 accumulate total loss weighted by batch size
        correct, total = _accuracy_from_logits(logits, yb)  # 3.12.26 compute batch accuracy counts
        running_correct += int(correct)  # 3.12.26 accumulate number of correct predictions
        running_total += int(total)  # 3.12.26 accumulate number of seen samples

    avg_loss = running_loss / max(1, running_total)  # 3.12.26 average evaluation loss across the epoch
    avg_acc = running_correct / max(1, running_total)  # 3.12.26 average evaluation accuracy across the epoch
    return avg_loss, avg_acc  # 3.12.26 return epoch evaluation metrics

def _to_uint8_rgb(img_hw3):  # 3.12.26 convert a float image in [0, 1] to uint8 for saving
    arr = np.asarray(img_hw3)  # 3.12.26 make sure the image is a NumPy array
    if arr.dtype != np.uint8:  # 3.12.26 convert non-uint8 images to uint8
        arr = np.clip(arr, 0.0, 1.0)  # 3.12.26 keep values in a valid image range
        arr = (arr * 255.0).round().astype(np.uint8)  # 3.12.26 scale [0, 1] floats to 8-bit image values
    return arr  # 3.12.26 return the uint8 RGB image

def save_misclassified_images(sids, y_true, y_pred, outdir, max_images=None):  # 3.12.26 save selected misclassified stimulus images for inspection
    outdir = Path(outdir)  # 3.12.26 normalize the output directory to a Path object
    outdir.mkdir(parents=True, exist_ok=True)  # 3.12.26 create the output directory if needed
    mism = np.where(np.asarray(y_true) != np.asarray(y_pred))[0]  # 3.12.26 find the positions of misclassified samples
    if max_images is not None:  # 3.12.26 optionally limit how many misclassified images are saved
        mism = mism[:int(max_images)]  # 3.12.26 keep only the first max_images mistakes

    rows = []  # 3.12.26 collect a small index table describing the saved images
    for j in mism:  # 3.12.26 loop over misclassified examples
        sid = int(sids[j])  # 3.12.26 stimulus ID for this misclassified sample
        true_i = int(y_true[j])  # 3.12.26 true class index
        pred_i = int(y_pred[j])  # 3.12.26 predicted class index
        true_lab = str(int_to_label[true_i])  # 3.12.26 true class name
        pred_lab = str(int_to_label[pred_i])  # 3.12.26 predicted class name

        img = get_img_sample(sid)  # 3.12.26 lazily load the original stimulus image
        img_u8 = _to_uint8_rgb(img)  # 3.12.26 convert the image to uint8 for saving
        fn = f"sid{sid:04d}_true-{true_lab}_pred-{pred_lab}.png"  # 3.12.26 descriptive output filename
        Image.fromarray(img_u8).save(str(outdir / fn))  # 3.12.26 save the stimulus image to disk

        rows.append({"sid": sid, "true": true_lab, "pred": pred_lab, "file": fn})  # 3.12.26 record this saved file in the index table

    pd.DataFrame(rows).to_csv(str(outdir / "misclassified_index.csv"), index=False)  # 3.12.26 save the image index table

@torch.no_grad()  # 3.12.26 prediction does not need gradient tracking
def predict_loader(model, dataloader, device=device, return_sids=False):  # 3.12.26 collect predictions from one DataLoader
    model.eval()  # 3.12.26 put the model in evaluation mode
    ys_true = []  # 3.12.26 collect true labels batch by batch
    ys_pred = []  # 3.12.26 collect predicted labels batch by batch
    non_block = (device.type == "cuda")  # 3.12.26 use non_blocking transfers only when running on GPU

    if return_sids:  # 3.12.26 optionally also return the stimulus IDs in DataLoader order
        ds_indices = np.asarray(dataloader.dataset.indices, dtype=np.int64)  # 3.12.26 the split indices stored in the dataset object
        sid_list = []  # 3.12.26 collect stimulus IDs batch by batch
        offset = 0  # 3.12.26 track where the next batch falls inside dataset.indices

    for xb, yb in dataloader:  # 3.12.26 iterate over batches in the DataLoader
        batch_size = int(yb.size(0))  # 3.12.26 number of samples in this batch
        xb = xb.to(device=device, dtype=torch.float32, non_blocking=non_block)  # 3.12.26 move inputs to the active device
        logits, _ = model(xb)  # 3.12.26 run the model and ignore the latent output here
        pred = torch.argmax(logits, dim=1).detach().cpu().numpy()  # 3.12.26 convert predicted class indices to NumPy
        ys_pred.append(pred)  # 3.12.26 store batch predictions
        ys_true.append(yb.detach().cpu().numpy())  # 3.12.26 store batch true labels

        if return_sids:  # 3.12.26 keep the matching stimulus IDs when requested
            sid_list.append(ds_indices[offset:offset + batch_size])  # 3.12.26 match this batch to its dataset indices
            offset += batch_size  # 3.12.26 move the index window to the next batch

    y_true_np = np.concatenate(ys_true).astype(np.int64)  # 3.12.26 concatenate all true labels into one array
    y_pred_np = np.concatenate(ys_pred).astype(np.int64)  # 3.12.26 concatenate all predicted labels into one array

    if return_sids:  # 3.12.26 optionally return stimulus IDs alongside labels
        sids_np = np.concatenate(sid_list).astype(np.int64)  # 3.12.26 concatenate all stimulus IDs into one array
        return y_true_np, y_pred_np, sids_np  # 3.12.26 return labels and stimulus IDs together

    return y_true_np, y_pred_np  # 3.12.26 return labels only

def compute_confusion_matrix_int(y_true, y_pred, n_classes):  # 3.12.26 make a stable integer confusion matrix with all classes present
    labels = list(range(int(n_classes)))  # 3.12.26 force a fixed label order from 0 to n_classes - 1
    cm = confusion_matrix(y_true, y_pred, labels=labels)  # 3.12.26 compute the confusion matrix in fixed class order
    return cm.astype(np.int64)  # 3.12.26 store the confusion matrix as int64

def save_confusion_matrix_bundle(y_true, y_pred, outdir, title="Confusion matrix", n_classes=None):  # 3.12.26 save numeric and plotted confusion-matrix artifacts
    outdir = Path(outdir)  # 3.12.26 normalize the output directory to a Path object
    outdir.mkdir(parents=True, exist_ok=True)  # 3.12.26 create the output directory if needed

    if n_classes is None:  # 3.12.26 default to the global class order from Cell 1
        n_classes = int(len(CLASS_ORDER))  # 3.12.26 use all classes in the notebook

    cm = compute_confusion_matrix_int(y_true, y_pred, n_classes=int(n_classes))  # 3.12.26 numeric confusion matrix in fixed class order
    np.save(str(outdir / "confusion_matrix.npy"), cm)  # 3.12.26 save the raw confusion matrix as a NumPy file

    cm_df = pd.DataFrame(cm, index=list(CLASS_ORDER), columns=list(CLASS_ORDER))  # 3.12.26 label the confusion matrix rows and columns with class names
    cm_df.to_csv(str(outdir / "confusion_matrix.csv"))  # 3.12.26 save the confusion matrix as a readable CSV file

    cm_meta = {  # 3.12.26 small metadata record describing the confusion matrix
        "class_order": list(CLASS_ORDER),  # 3.12.26 save the class-label order used in the matrix
        "shape": [int(cm.shape[0]), int(cm.shape[1])],  # 3.12.26 save the confusion-matrix shape
    }  # 3.12.26 end confusion-matrix metadata dictionary
    with open(str(outdir / "confusion_matrix_meta.json"), "w") as f:  # 3.12.26 save the confusion-matrix metadata as JSON
        json.dump(cm_meta, f, indent=2)  # 3.12.26 write the metadata JSON file

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(CLASS_ORDER))  # 3.12.26 display object for plotting the confusion matrix
    fig, ax = plt.subplots(figsize=(6, 6))  # 3.12.26 make a square figure for the confusion-matrix plot
    disp.plot(ax=ax, xticks_rotation=45, colorbar=False)  # 3.12.26 draw the confusion matrix on the axes
    ax.set_title(str(title))  # 3.12.26 add a title describing this confusion matrix
    fig.tight_layout()  # 3.12.26 reduce clipping in the saved figure
    fig.savefig(str(outdir / "confusion_matrix.png"), dpi=200)  # 3.12.26 save the plotted confusion matrix as a PNG
    plt.close(fig)  # 3.12.26 close the figure to free memory

    return cm  # 3.12.26 return the numeric confusion matrix for convenience

def save_predictions_npz(y_true, y_pred, sids, outdir, prefix="val"):  # 3.12.26 save full prediction arrays for later aggregation
    outdir = Path(outdir)  # 3.12.26 normalize the output directory to a Path object
    outdir.mkdir(parents=True, exist_ok=True)  # 3.12.26 create the output directory if needed
    np.savez_compressed(  # 3.12.26 save prediction arrays in one compressed file
        str(outdir / f"{prefix}_preds.npz"),  # 3.12.26 output file path
        y_true=np.asarray(y_true, dtype=np.int64),  # 3.12.26 true labels
        y_pred=np.asarray(y_pred, dtype=np.int64),  # 3.12.26 predicted labels
        sids=np.asarray(sids, dtype=np.int64),  # 3.12.26 stimulus IDs
        class_order=np.asarray(list(CLASS_ORDER), dtype=object),  # 3.12.26 class-label order for downstream reading
    )

def train_model(  # 3.12.26 main training loop used later in the notebook
    model,
    dl_train,
    dl_val,
    optimizer,
    criterion,
    epochs=10,
    save_best=False,
    ckpt_path="best_model.pt",
    device=device,
    grad_clip=None,
    save_val_artifacts_epochs=None,
    artifacts_dir=RUN_ARTIFACTS_DIR,
    max_misclass_images=None,
):
    history = {  # 3.12.26 collect all main outputs from one training run
        "train_loss": [],  # 3.12.26 per-epoch training loss
        "train_acc": [],  # 3.12.26 per-epoch training accuracy
        "val_loss": [],  # 3.12.26 per-epoch validation loss
        "val_acc": [],  # 3.12.26 per-epoch validation accuracy
        "epoch_sec": [],  # 3.12.26 per-epoch elapsed time
        "val_artifact_dirs": {},  # 3.12.26 record where validation artifacts were saved
    }  # 3.12.26 end history dictionary

    save_trial_val_artifacts = (save_val_artifacts_epochs is not None) and (len(set(save_val_artifacts_epochs)) > 0)  # 3.12.26 treat a non-empty epoch set as a simple enable flag
    best_val_acc = -1.0  # 3.12.26 best validation accuracy seen so far
    best_epoch = -1  # 3.12.26 epoch number that achieved the best validation accuracy

    ckpt_path = Path(ckpt_path)  # 3.12.26 normalize checkpoint path to a Path object
    ckpt_path.parent.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the checkpoint folder exists before saving weights

    for epoch in range(1, epochs + 1):  # 3.12.26 loop over training epochs
        t0 = time.time()  # 3.12.26 start timing this epoch

        train_loss, train_acc = train_one_epoch(  # 3.12.26 run one training epoch
            model,
            dl_train,
            optimizer,
            criterion,
            device=device,
            grad_clip=grad_clip,
        )
        val_loss, val_acc = eval_one_epoch(  # 3.12.26 run one validation epoch
            model,
            dl_val,
            criterion,
            device=device,
        )

        history["train_loss"].append(float(train_loss))  # 3.12.26 store training loss for this epoch
        history["train_acc"].append(float(train_acc))  # 3.12.26 store training accuracy for this epoch
        history["val_loss"].append(float(val_loss))  # 3.12.26 store validation loss for this epoch
        history["val_acc"].append(float(val_acc))  # 3.12.26 store validation accuracy for this epoch
        history["epoch_sec"].append(float(time.time() - t0))  # 3.12.26 store elapsed time for this epoch

        print(  # 3.12.26 print a concise epoch summary
            f"Epoch {epoch:02d}/{epochs} | "
            f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.4f} | "
            f"{history['epoch_sec'][-1]:.1f}s"
        )

        if save_best and (val_acc > best_val_acc):  # 3.12.26 update and save the best checkpoint when validation improves
            best_val_acc = float(val_acc)  # 3.12.26 record the new best validation accuracy
            best_epoch = int(epoch)  # 3.12.26 record the epoch that produced the best validation accuracy
            torch.save(model.state_dict(), str(ckpt_path))  # 3.12.26 save best-model weights
            print(f"  saved best checkpoint -> {ckpt_path} (val acc {best_val_acc:.4f})")  # 3.12.26 confirm best checkpoint saving

    final_ckpt_path = ckpt_path.with_name("final_model.pt")  # 3.12.26 keep the final checkpoint next to the best checkpoint
    torch.save(model.state_dict(), str(final_ckpt_path))  # 3.12.26 save the final model weights after the last epoch
    print(f"  saved final checkpoint -> {final_ckpt_path}")  # 3.12.26 confirm final checkpoint saving

    if save_trial_val_artifacts:  # 3.12.26 save validation artifacts once per trial after training finishes
        n_cls = int(len(CLASS_ORDER))  # 3.12.26 total number of classes for confusion-matrix saving
        snap_name = "final"  # 3.12.26 default artifact snapshot is the final model
        snap_epoch = int(epochs)  # 3.12.26 default artifact epoch is the final epoch
        loaded_best = False  # 3.12.26 track whether best weights were temporarily loaded for artifact saving

        if save_best and (best_epoch >= 1):  # 3.12.26 prefer the best checkpoint when one exists
            model.load_state_dict(torch.load(str(ckpt_path), map_location=device))  # 3.12.26 load best weights for validation-artifact saving
            snap_name = "best"  # 3.12.26 mark the snapshot as best
            snap_epoch = int(best_epoch)  # 3.12.26 record which epoch produced the best checkpoint
            loaded_best = True  # 3.12.26 remember that best weights were loaded

        out_dir = Path(artifacts_dir) / f"val_{snap_name}_epoch{int(snap_epoch):02d}"  # 3.12.26 make one validation-artifact folder for this trial
        out_dir.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the validation-artifact folder exists
        history["val_artifact_dirs"][str(snap_name)] = str(out_dir)  # 3.12.26 record where validation artifacts were saved

        y_true_np, y_pred_np, sids_np = predict_loader(  # 3.12.26 collect all validation predictions once
            model,
            dl_val,
            device=device,
            return_sids=True,
        )

        save_confusion_matrix_bundle(  # 3.12.26 save numeric and plotted confusion-matrix artifacts
            y_true_np,
            y_pred_np,
            outdir=out_dir,
            title=f"Val confusion matrix ({snap_name}, epoch {int(snap_epoch):02d})",
            n_classes=int(n_cls),
        )

        save_predictions_npz(  # 3.12.26 save full validation predictions for later aggregation
            y_true_np,
            y_pred_np,
            sids_np,
            outdir=out_dir,
            prefix=f"val_{snap_name}",
        )

        if (max_misclass_images is not None) and (int(max_misclass_images) > 0):  # 3.12.26 save misclassified images only when the user explicitly asks for them
            save_misclassified_images(  # 3.12.26 save a capped number of misclassified stimulus images
                sids_np,
                y_true_np,
                y_pred_np,
                out_dir / "misclassified",
                max_images=int(max_misclass_images),
            )

        val_acc_from_preds = float((np.asarray(y_true_np) == np.asarray(y_pred_np)).mean())  # 3.12.26 compute validation accuracy directly from saved predictions
        val_summary = {  # 3.12.26 small summary record for this validation-artifact snapshot
            "snapshot": str(snap_name),  # 3.12.26 best or final
            "epoch": int(snap_epoch),  # 3.12.26 epoch number for the saved snapshot
            "best_epoch": int(best_epoch),  # 3.12.26 epoch number with the best validation accuracy
            "best_val_acc_reported": float(best_val_acc),  # 3.12.26 best validation accuracy recorded during training
            "val_acc_from_preds": float(val_acc_from_preds),  # 3.12.26 validation accuracy recomputed from saved predictions
            "n_val": int(len(y_true_np)),  # 3.12.26 number of validation samples
            "class_order": list(CLASS_ORDER),  # 3.12.26 class-label order for downstream reading
        }  # 3.12.26 end validation-summary dictionary
        with open(str(out_dir / "val_summary.json"), "w") as f:  # 3.12.26 save the validation summary as JSON
            json.dump(val_summary, f, indent=2)  # 3.12.26 write the validation summary file

        if loaded_best:  # 3.12.26 restore the final weights after temporarily loading the best checkpoint
            model.load_state_dict(torch.load(str(final_ckpt_path), map_location=device))  # 3.12.26 restore the final model state

    history["best_val_acc"] = float(best_val_acc)  # 3.12.26 save the best validation accuracy in the returned history
    history["best_epoch"] = int(best_epoch)  # 3.12.26 save the best epoch number in the returned history
    history["ckpt_path"] = str(ckpt_path) if save_best else None  # 3.12.26 save the best-checkpoint path when best-model saving is enabled
    history["final_ckpt_path"] = str(final_ckpt_path)  # 3.12.26 save the final-checkpoint path in the returned history

    return history  # 3.12.26 return all saved metrics and paths for this training run

In [ ]:
# Cell 7 — Run CNN experiments for BUILD_GROUPS with weighted multi-split + multi-init trials + final test eval  # 03.14.2026 update header to match weighted Cell 3 logic

EPOCHS = 10  # 3.12.26 number of training epochs per trial
LR = 1e-4  # 3.12.26 learning rate for the Adam optimizer
BATCH_SIZE = 20  # 3.12.26 batch size used when building the real training DataLoaders in this cell
NUM_WORKERS = 0  # 3.12.26 number of DataLoader worker processes

N_SPLITS = 2  # 3.12.26 number of different balanced train/val/test splits to run
N_INIT_TRIALS = 2  # 3.12.26 number of repeated model initializations per split and channel group

KEEP_SHUFFLE_IDENTICAL_ACROSS_GROUPS = True  # 3.12.26 keep training minibatch order the same across groups within a split
MAX_MISCLASS_IMAGES_SAVE = None  # 3.12.26 do not save misclassified images unless you set a positive cap
SAVE_TEST_ARTIFACTS = True  # 3.12.26 save numeric and plotted test artifacts for each trial
SAVE_TEST_MISCLASS_IMAGES = False  # 3.12.26 keep heavy test-image saving off by default
GRAD_CLIP = None  # 3.12.26 optional gradient clipping threshold; leave as None for no clipping
PIN_MEMORY = (device.type == "cuda")  # 3.12.26 pin DataLoader memory only when using a GPU

assert "BUILD_GROUPS" in globals(), "Run Cell 1 first so BUILD_GROUPS is defined."  # 3.12.26 fail fast if channel-group settings are missing
assert "CHANNEL_GROUPS" in globals(), "Run Cell 1 first so CHANNEL_GROUPS is defined."  # 3.12.26 fail fast if channel definitions are missing
assert "TARGET_CLASS_WEIGHTS" in globals(), "Run Cell 1 first so TARGET_CLASS_WEIGHTS is defined."  # 03.14.2026 require the weighted class targets from Cell 1
assert "keys_order" in globals(), "Run Cell 2 first so keys_order is defined."  # 3.12.26 fail fast if the saved response-channel order is missing
assert "y" in globals(), "Run Cell 3 first so integer labels y are defined."  # 3.12.26 fail fast if class labels are missing
assert "meta" in globals(), "Run Cell 2 first so the metadata table is defined."  # 3.12.26 fail fast if metadata is missing
assert "balance_bin" in meta.columns, "Run Cell 3 first so balance_bin exists in meta."  # 3.12.26 require the split-balance key from Cell 3
assert "TRAIN_FRAC" in globals() and "VAL_FRAC" in globals() and "TEST_FRAC" in globals(), "Run Cell 3 first so split fractions are defined."  # 3.12.26 require the split proportions from Cell 3
assert "RetinaHueDataset" in globals(), "Run Cell 4 first so RetinaHueDataset is defined."  # 3.12.26 require the dataset class from Cell 4
assert "build_seeded_model" in globals(), "Run Cell 5 first so build_seeded_model is defined."  # 3.12.26 require the seeded model builder from Cell 5
assert "train_model" in globals(), "Run Cell 6 first so train_model is defined."  # 3.12.26 require the training loop from Cell 6
assert "eval_one_epoch" in globals(), "Run Cell 6 first so eval_one_epoch is defined."  # 3.12.26 require the evaluation loop from Cell 6
assert "make_optimizer" in globals(), "Run Cell 6 first so make_optimizer is defined."  # 3.12.26 require the optimizer helper from Cell 6
assert "make_criterion" in globals(), "Run Cell 6 first so make_criterion is defined."  # 3.12.26 require the loss-function helper from Cell 6
assert "predict_loader" in globals(), "Run Cell 6 first so predict_loader is defined."  # 3.12.26 require the prediction helper from Cell 6
assert "save_confusion_matrix_bundle" in globals(), "Run Cell 6 first so save_confusion_matrix_bundle is defined."  # 3.12.26 require confusion-matrix saving utilities
assert "save_predictions_npz" in globals(), "Run Cell 6 first so save_predictions_npz is defined."  # 3.12.26 require prediction saving utilities
assert "get_outs_fill_sample" in globals(), "Run Cell 2 first so get_outs_fill_sample is defined."  # 3.12.26 require lazy access to the filled response maps
assert "make_torch_generator" in globals(), "Run Cell 1 first so make_torch_generator is defined."  # 3.12.26 require deterministic DataLoader generators
assert "seed_worker" in globals(), "Run Cell 1 first so seed_worker is defined."  # 3.12.26 require deterministic worker seeding
assert abs(float(TRAIN_FRAC) + float(VAL_FRAC) + float(TEST_FRAC) - 1.0) < 1e-9, "TRAIN_FRAC + VAL_FRAC + TEST_FRAC must sum to 1.0."  # 3.12.26 re-check split fractions before long runs

RUN_ARTIFACTS_DIR = ARTIFACTS_DIR / "multi_run_experiments"  # 3.12.26 keep all real experiment outputs under one clear root folder
RUN_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the experiment artifact folder exists before training begins

def allocate_three_way(n, train_frac, val_frac, test_frac):  # 3.12.26 helper to convert fractional split targets into exact integer counts
    quotas = np.array([n * train_frac, n * val_frac, n * test_frac], dtype=float)  # 3.12.26 ideal non-integer split counts
    base = np.floor(quotas).astype(int)  # 3.12.26 start from floor counts
    need = int(n - base.sum())  # 3.12.26 number of leftover samples still needing assignment
    if need > 0:  # 3.12.26 distribute leftovers to the largest fractional remainders
        order = np.argsort(-(quotas - base))  # 3.12.26 largest fractional parts first
        for pos in order[:need]:  # 3.12.26 give one extra sample to the needed positions
            base[pos] += 1  # 3.12.26 update the integer allocation
    return int(base[0]), int(base[1]), int(base[2])  # 3.12.26 return train, val, and test counts

def counts_by_hue_local(idxs):  # 3.12.26 helper to count hues in a fixed class order
    return meta.loc[np.asarray(idxs, dtype=np.int64), "hue"].astype(str).value_counts().reindex(CLASS_ORDER).fillna(0).astype(int)  # 3.12.26 return hue counts in CLASS_ORDER

def make_balanced_pool_indices(split_seed):  # 03.14.2026 rebuild the weighted pool for one split seed
    rng = np.random.default_rng(int(split_seed))  # 3.12.26 split-specific random generator
    all_idx_local = np.arange(len(meta), dtype=np.int64)  # 3.12.26 all dataset indices in row order
    y_str_local = meta["hue"].astype(str).to_numpy()  # 3.12.26 hue labels in dataset order
    avail_counts_local = meta["hue"].astype(str).value_counts().reindex(CLASS_ORDER).fillna(0).astype(int)  # 3.12.26 available count for each hue
    target_weights_local = pd.Series(TARGET_CLASS_WEIGHTS).reindex(CLASS_ORDER).astype(float)  # 03.14.2026 use the weighted class targets from Cell 1
    assert target_weights_local.notna().all(), f"Missing TARGET_CLASS_WEIGHTS entries for: {target_weights_local[target_weights_local.isna()].index.tolist()}"  # 03.14.2026 require one weight for each class
    min_weight_local = float(target_weights_local[target_weights_local > 0].min())  # 03.14.2026 smallest positive weight defines one ratio unit
    weight_units_local = (target_weights_local / min_weight_local).round().astype(int)  # 03.14.2026 convert 25/25/12.5/... into integer ratio units
    assert np.allclose(target_weights_local / min_weight_local, weight_units_local.astype(float)), "TARGET_CLASS_WEIGHTS must reduce cleanly to integer ratio units."  # 03.14.2026 require exact ratio-style weights
    pool_unit_local = int(min(avail_counts_local[hue] // weight_units_local[hue] for hue in CLASS_ORDER))  # 03.14.2026 largest shared unit that fits all classes
    target_pool_counts_local = pd.Series(  # 03.14.2026 exact per-class counts for the weighted pool
        {hue: int(weight_units_local[hue] * pool_unit_local) for hue in CLASS_ORDER},  # 03.14.2026 build the weighted pool counts from ratio units
        index=CLASS_ORDER,  # 03.14.2026 keep class order stable
        dtype=int,  # 03.14.2026 store pool counts as integers
    )  # 03.14.2026 end weighted pool-count series

    pool_parts_local = []  # 03.14.2026 collect one weighted subset per hue
    for hue in CLASS_ORDER:  # 03.14.2026 sample the exact weighted count from each hue
        idx_h = all_idx_local[y_str_local == hue].copy()  # 3.12.26 indices for one hue
        rng.shuffle(idx_h)  # 3.12.26 randomize before taking the balanced subset
        pool_parts_local.append(idx_h[:int(target_pool_counts_local[hue])])  # 03.14.2026 keep the exact weighted count for this hue

    pool_idx_local = np.concatenate(pool_parts_local).astype(np.int64)  # 03.14.2026 final weighted pool for this split seed
    rng.shuffle(pool_idx_local)  # 3.12.26 mix hues and conditions within the balanced pool
    return pool_idx_local, avail_counts_local, target_pool_counts_local  # 03.14.2026 return the weighted pool and its class counts

def make_one_balanced_split(split_seed):  # 03.14.2026 build one weighted train/val/test split using the revised Cell 3 logic
    rng = np.random.default_rng(int(split_seed))  # 3.12.26 split-specific random generator
    pool_idx_local, avail_counts_local, target_pool_counts_local = make_balanced_pool_indices(split_seed=int(split_seed))  # 03.14.2026 build the weighted pool for this split
    split_fracs_local = {"train": float(TRAIN_FRAC), "val": float(VAL_FRAC), "test": float(TEST_FRAC)}  # 3.12.26 local copy of split fractions
    y_str_local = meta["hue"].astype(str).to_numpy()  # 3.12.26 hue labels in dataset order

    train_parts_local = []  # 3.12.26 collect train indices for each hue
    val_parts_local = []  # 3.12.26 collect validation indices for each hue
    test_parts_local = []  # 3.12.26 collect test indices for each hue
    bin_to_alloc_local = {}  # 3.12.26 record how each hue-specific balance bin was allocated
    hue_target_counts_local = {}  # 3.12.26 record the exact train/val/test counts for each hue

    for hue in CLASS_ORDER:  # 3.12.26 split each hue separately so class balance stays exact
        idx_h = pool_idx_local[y_str_local[pool_idx_local] == hue].astype(np.int64)  # 3.12.26 balanced-pool indices for this hue only
        n_train_h, n_val_h, n_test_h = allocate_three_way(len(idx_h), float(TRAIN_FRAC), float(VAL_FRAC), float(TEST_FRAC))  # 3.12.26 exact split sizes for this hue
        hue_target_counts_local[hue] = {"train": int(n_train_h), "val": int(n_val_h), "test": int(n_test_h)}  # 3.12.26 save the per-hue split targets

        assigned = {"train": [], "val": [], "test": []}  # 3.12.26 track assigned indices for this hue
        bin_labels_h = meta.loc[idx_h, "balance_bin"].astype(str).to_numpy()  # 3.12.26 balance-bin labels for this hue
        bin_sizes = pd.Series(bin_labels_h).value_counts()  # 3.12.26 count how many samples fall in each balance bin
        ordered_bins = sorted(bin_sizes.index.tolist(), key=lambda b: (int(bin_sizes.loc[b]), str(b)))  # 3.12.26 process rare bins first, then larger bins

        bin_to_indices_local = {}  # 03.14.2026 store shuffled indices once per hue-bin
        per_bin_counts_local = {}  # 03.14.2026 store base split counts for each hue-bin
        per_bin_frac_parts_local = {}  # 03.14.2026 store fractional remainders for each hue-bin
        base_totals_local = {"train": 0, "val": 0, "test": 0}  # 03.14.2026 total base counts before leftover placement

        for bin_name in ordered_bins:  # 03.14.2026 first pass: compute proportional base allocations for each hue-bin
            idx_bin = idx_h[bin_labels_h == bin_name].astype(np.int64)  # 03.14.2026 pooled indices in this hue-specific balance bin
            rng.shuffle(idx_bin)  # 03.14.2026 shuffle within the bin before final slicing
            bin_to_indices_local[bin_name] = idx_bin  # 03.14.2026 save shuffled indices for the second pass

            quotas = np.array([len(idx_bin) * TRAIN_FRAC, len(idx_bin) * VAL_FRAC, len(idx_bin) * TEST_FRAC], dtype=float)  # 03.14.2026 ideal per-bin split quotas
            base = np.floor(quotas).astype(int)  # 03.14.2026 base per-bin counts from floor allocation

            per_bin_counts_local[bin_name] = {"train": int(base[0]), "val": int(base[1]), "test": int(base[2])}  # 03.14.2026 save base counts for this hue-bin
            per_bin_frac_parts_local[bin_name] = {  # 03.14.2026 save fractional remainders for leftover assignment
                "train": float(quotas[0] - base[0]),  # 03.14.2026 train fractional remainder
                "val": float(quotas[1] - base[1]),  # 03.14.2026 val fractional remainder
                "test": float(quotas[2] - base[2]),  # 03.14.2026 test fractional remainder
            }  # 03.14.2026 end remainder record for this hue-bin
            base_totals_local["train"] += int(base[0])  # 03.14.2026 accumulate base train count across hue-bins
            base_totals_local["val"] += int(base[1])  # 03.14.2026 accumulate base val count across hue-bins
            base_totals_local["test"] += int(base[2])  # 03.14.2026 accumulate base test count across hue-bins

        leftover_targets_local = {  # 03.14.2026 exact extra counts each split still needs after base allocations
            "train": int(n_train_h - base_totals_local["train"]),  # 03.14.2026 train leftovers still needed for this hue
            "val": int(n_val_h - base_totals_local["val"]),  # 03.14.2026 val leftovers still needed for this hue
            "test": int(n_test_h - base_totals_local["test"]),  # 03.14.2026 test leftovers still needed for this hue
        }  # 03.14.2026 end leftover-target dictionary
        assert all(v >= 0 for v in leftover_targets_local.values()), f"Negative leftover target for hue {hue}: {leftover_targets_local}"  # 03.14.2026 require valid leftover targets

        for bin_name in ordered_bins:  # 03.14.2026 second pass: place leftovers while preserving exact hue totals
            idx_bin = bin_to_indices_local[bin_name]  # 03.14.2026 recover shuffled indices for this hue-bin
            alloc_counts = dict(per_bin_counts_local[bin_name])  # 03.14.2026 start from the base per-bin counts
            leftovers = int(len(idx_bin) - sum(alloc_counts.values()))  # 03.14.2026 number of unassigned samples left in this hue-bin

            priority = sorted(  # 03.14.2026 prefer largest fractional remainder, then largest remaining need
                ["train", "val", "test"],  # 03.14.2026 fixed split order for stable priority sorting
                key=lambda s: (-per_bin_frac_parts_local[bin_name][s], -leftover_targets_local[s], -split_fracs_local[s], s),  # 03.14.2026 stable remainder-first priority
            )  # 03.14.2026 end primary priority order
            for split_name in priority:  # 03.14.2026 assign leftovers one by one within this hue-bin
                if leftovers == 0:  # 03.14.2026 stop once this hue-bin is fully assigned
                    break  # 03.14.2026 no more leftovers to place in this hue-bin
                if leftover_targets_local[split_name] > 0:  # 03.14.2026 only give extras to splits that still need them
                    alloc_counts[split_name] += 1  # 03.14.2026 add one leftover sample to this split
                    leftover_targets_local[split_name] -= 1  # 03.14.2026 reduce the remaining need for this split
                    leftovers -= 1  # 03.14.2026 reduce the leftover count for this hue-bin

            if leftovers > 0:  # 03.14.2026 fallback if any leftovers remain after remainder-based assignment
                fallback_priority = sorted(  # 03.14.2026 use remaining split need as the primary fallback rule
                    ["train", "val", "test"],  # 03.14.2026 fixed split order for stable fallback sorting
                    key=lambda s: (-leftover_targets_local[s], -per_bin_frac_parts_local[bin_name][s], -split_fracs_local[s], s),  # 03.14.2026 stable need-first fallback priority
                )  # 03.14.2026 end fallback priority order
                for split_name in fallback_priority:  # 03.14.2026 continue assigning any remaining leftovers
                    if leftovers == 0:  # 03.14.2026 stop once this hue-bin is fully assigned
                        break  # 03.14.2026 no more leftovers to place in this hue-bin
                    if leftover_targets_local[split_name] > 0:  # 03.14.2026 only give extras to splits that still need them
                        alloc_counts[split_name] += 1  # 03.14.2026 add one leftover sample to this split
                        leftover_targets_local[split_name] -= 1  # 03.14.2026 reduce the remaining need for this split
                        leftovers -= 1  # 03.14.2026 reduce the leftover count for this hue-bin

            assert leftovers == 0, f"Could not place all leftovers for hue {hue}, bin {bin_name}"  # 03.14.2026 require full assignment within each hue-bin

            start = 0  # 03.14.2026 running pointer into the shuffled indices for this hue-bin
            for split_name in ["train", "val", "test"]:  # 03.14.2026 slice this hue-bin into train, val, and test chunks
                n_take = int(alloc_counts[split_name])  # 03.14.2026 number of samples to take for this split
                assigned[split_name].extend(idx_bin[start:start + n_take].tolist())  # 03.14.2026 append this split's slice from the hue-bin
                start += n_take  # 03.14.2026 advance to the next slice boundary

            bin_to_alloc_local[f"{hue}|{bin_name}"] = {  # 3.12.26 save how this hue-specific bin was allocated
                "n": int(len(idx_bin)),  # 3.12.26 number of pooled samples in this bin
                "n_train": int(alloc_counts["train"]),  # 3.12.26 number sent to train
                "n_val": int(alloc_counts["val"]),  # 3.12.26 number sent to validation
                "n_test": int(alloc_counts["test"]),  # 3.12.26 number sent to test
            }  # 3.12.26 end allocation record for this bin

        assert all(v == 0 for v in leftover_targets_local.values()), f"Leftover targets not exhausted for hue {hue}: {leftover_targets_local}"  # 03.14.2026 require exact hue totals after hue-bin allocation
        assert len(assigned["train"]) == int(n_train_h), f"Train count mismatch for hue {hue}"  # 3.12.26 require the exact target train count for this hue
        assert len(assigned["val"]) == int(n_val_h), f"Val count mismatch for hue {hue}"  # 3.12.26 require the exact target validation count for this hue
        assert len(assigned["test"]) == int(n_test_h), f"Test count mismatch for hue {hue}"  # 3.12.26 require the exact target test count for this hue

        train_parts_local.append(np.array(assigned["train"], dtype=np.int64))  # 3.12.26 store this hue's train indices
        val_parts_local.append(np.array(assigned["val"], dtype=np.int64))  # 3.12.26 store this hue's validation indices
        test_parts_local.append(np.array(assigned["test"], dtype=np.int64))  # 3.12.26 store this hue's test indices

    train_idx_local = np.concatenate(train_parts_local).astype(np.int64)  # 3.12.26 final train indices for this split seed
    val_idx_local = np.concatenate(val_parts_local).astype(np.int64)  # 3.12.26 final validation indices for this split seed
    test_idx_local = np.concatenate(test_parts_local).astype(np.int64)  # 3.12.26 final test indices for this split seed

    rng.shuffle(train_idx_local)  # 3.12.26 randomize final train order
    rng.shuffle(val_idx_local)  # 3.12.26 randomize final validation order
    rng.shuffle(test_idx_local)  # 3.12.26 randomize final test order

    assert len(set(train_idx_local) & set(val_idx_local)) == 0, "Train and val splits overlap."  # 3.12.26 require non-overlapping splits
    assert len(set(train_idx_local) & set(test_idx_local)) == 0, "Train and test splits overlap."  # 3.12.26 require non-overlapping splits
    assert len(set(val_idx_local) & set(test_idx_local)) == 0, "Val and test splits overlap."  # 3.12.26 require non-overlapping splits
    assert int(len(train_idx_local) + len(val_idx_local) + len(test_idx_local)) == int(len(pool_idx_local)), "Split sizes must sum to the balanced pool size."  # 3.12.26 require full coverage of the balanced pool

    split_info_local = {  # 3.12.26 compact summary of this split for saving and debugging
        "split_seed": int(split_seed),  # 3.12.26 split seed used to build this split
        "avail_counts": avail_counts_local.to_dict(),  # 3.12.26 full-dataset class counts
        "target_class_weights": dict(TARGET_CLASS_WEIGHTS),  # 03.14.2026 save the requested weighted class targets
        "target_pool_counts": target_pool_counts_local.to_dict(),  # 03.14.2026 save the exact weighted pool counts used for this split
        "pool_size": int(len(pool_idx_local)),  # 3.12.26 total balanced-pool size
        "pool_counts": counts_by_hue_local(pool_idx_local).to_dict(),  # 3.12.26 hue counts in the balanced pool
        "train_size": int(len(train_idx_local)),  # 3.12.26 train split size
        "val_size": int(len(val_idx_local)),  # 3.12.26 validation split size
        "test_size": int(len(test_idx_local)),  # 3.12.26 test split size
        "train_counts": counts_by_hue_local(train_idx_local).to_dict(),  # 3.12.26 hue counts in train
        "val_counts": counts_by_hue_local(val_idx_local).to_dict(),  # 3.12.26 hue counts in validation
        "test_counts": counts_by_hue_local(test_idx_local).to_dict(),  # 3.12.26 hue counts in test
        "hue_target_counts": hue_target_counts_local,  # 3.12.26 exact target sizes used for each hue
        "n_unique_train_bins": int(meta.loc[train_idx_local, "bin_id"].nunique()),  # 3.12.26 number of hue+condition bins present in train
        "n_unique_val_bins": int(meta.loc[val_idx_local, "bin_id"].nunique()),  # 3.12.26 number of hue+condition bins present in validation
        "n_unique_test_bins": int(meta.loc[test_idx_local, "bin_id"].nunique()),  # 3.12.26 number of hue+condition bins present in test
        "bin_allocations": bin_to_alloc_local,  # 3.12.26 per-bin allocation record for reproducibility
    }  # 3.12.26 end split-summary dictionary

    return train_idx_local, val_idx_local, test_idx_local, split_info_local  # 3.12.26 return split indices and summary information

def make_loaders_for_group(group_name, train_idx_use, val_idx_use, test_idx_use, shuffle_seed):  # 3.12.26 build split-specific DataLoaders for one channel group
    if group_name not in CHANNEL_GROUPS:  # 3.12.26 validate the requested group name
        raise ValueError(f"group_name must be one of {list(CHANNEL_GROUPS.keys())}")  # 3.12.26 stop early if the group name is invalid

    keys_subset = CHANNEL_GROUPS[group_name]  # 3.12.26 response-channel names used by this group
    key_to_idx = {k: i for i, k in enumerate(keys_order)}  # 3.12.26 map saved response keys to channel indices
    missing_keys = [k for k in keys_subset if k not in key_to_idx]  # 3.12.26 detect any requested channels missing from the dataset
    assert len(missing_keys) == 0, f"Missing channels in keys_order for {group_name}: {missing_keys}"  # 3.12.26 fail early if a requested channel is absent

    channel_idxs = np.array([key_to_idx[k] for k in keys_subset], dtype=np.int64)  # 3.12.26 integer response-channel indices for this group
    c_run = int(len(channel_idxs))  # 3.12.26 number of CNN input channels for this group

    ds_train = RetinaHueDataset(indices=train_idx_use, labels=y, channel_idxs=channel_idxs)  # 3.12.26 training dataset for this split and group
    ds_val = RetinaHueDataset(indices=val_idx_use, labels=y, channel_idxs=channel_idxs)  # 3.12.26 validation dataset for this split and group
    ds_test = RetinaHueDataset(indices=test_idx_use, labels=y, channel_idxs=channel_idxs)  # 3.12.26 test dataset for this split and group

    dl_train_generator = make_torch_generator(int(shuffle_seed))  # 3.12.26 deterministic generator for training-batch shuffling

    dl_train = DataLoader(  # 3.12.26 training DataLoader for this split and group
        ds_train,
        batch_size=int(BATCH_SIZE),
        shuffle=True,
        num_workers=int(NUM_WORKERS),
        pin_memory=bool(PIN_MEMORY),
        generator=dl_train_generator,
        worker_init_fn=seed_worker if int(NUM_WORKERS) > 0 else None,
    )
    dl_val = DataLoader(  # 3.12.26 validation DataLoader for this split and group
        ds_val,
        batch_size=int(BATCH_SIZE),
        shuffle=False,
        num_workers=int(NUM_WORKERS),
        pin_memory=bool(PIN_MEMORY),
        worker_init_fn=seed_worker if int(NUM_WORKERS) > 0 else None,
    )
    dl_test = DataLoader(  # 3.12.26 test DataLoader for this split and group
        ds_test,
        batch_size=int(BATCH_SIZE),
        shuffle=False,
        num_workers=int(NUM_WORKERS),
        pin_memory=bool(PIN_MEMORY),
        worker_init_fn=seed_worker if int(NUM_WORKERS) > 0 else None,
    )

    return keys_subset, channel_idxs, c_run, ds_train, ds_val, ds_test, dl_train, dl_val, dl_test  # 3.12.26 return all split/group-specific loader objects

def make_criterion_for_split(train_idx_use):  # 3.12.26 build the loss function for one split using only the training indices
    return make_criterion(  # 3.12.26 delegate class-weight logic to the helper from Cell 6
        y_int=y,
        idxs=train_idx_use,
        n_classes=int(n_classes),
        device=device,
        use_class_weights=bool(USE_CLASS_WEIGHTS),
    )

def evaluate_checkpoint_on_test(model, ckpt_path, dl_test, criterion, trial_art_dir, snapshot_name):  # 3.12.26 evaluate one saved checkpoint on the test split
    model.load_state_dict(torch.load(str(ckpt_path), map_location=device))  # 3.12.26 load the requested checkpoint into the model
    test_loss, test_acc = eval_one_epoch(model=model, dataloader=dl_test, criterion=criterion, device=device)  # 3.12.26 compute test loss and accuracy for this checkpoint
    print(f"TEST({snapshot_name}) | loss {test_loss:.4f} | acc {test_acc:.4f}")  # 3.12.26 print a concise test summary for this checkpoint

    test_art_dir = None  # 3.12.26 default to no saved test-artifact directory
    if bool(SAVE_TEST_ARTIFACTS):  # 3.12.26 optionally save numeric and plotted test artifacts
        y_true_t, y_pred_t, sids_t = predict_loader(model, dl_test, device=device, return_sids=True)  # 3.12.26 collect full test predictions and stimulus IDs
        test_art_dir = Path(trial_art_dir) / f"test_{snapshot_name}"  # 3.12.26 separate best and final test artifacts into different folders
        test_art_dir.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the test-artifact folder exists

        save_confusion_matrix_bundle(  # 3.12.26 save numeric and plotted confusion-matrix artifacts for the test split
            y_true_t,
            y_pred_t,
            outdir=test_art_dir,
            title=f"Test confusion matrix ({snapshot_name})",
            n_classes=int(n_classes),
        )
        save_predictions_npz(  # 3.12.26 save full test predictions for later aggregation
            y_true_t,
            y_pred_t,
            sids_t,
            outdir=test_art_dir,
            prefix=f"test_{snapshot_name}",
        )

        if bool(SAVE_TEST_MISCLASS_IMAGES) and (MAX_MISCLASS_IMAGES_SAVE is not None) and (int(MAX_MISCLASS_IMAGES_SAVE) > 0):  # 3.12.26 save misclassified test images only if explicitly requested
            save_misclassified_images(  # 3.12.26 save a capped number of misclassified test stimulus images
                sids_t,
                y_true_t,
                y_pred_t,
                test_art_dir / "misclassified",
                max_images=int(MAX_MISCLASS_IMAGES_SAVE),
            )

    return float(test_loss), float(test_acc), (str(test_art_dir) if test_art_dir is not None else None)  # 3.12.26 return test metrics and artifact location

print("Running BUILD_GROUPS:", BUILD_GROUPS)  # 3.12.26 show which channel groups will be trained
print("Split seed base:", int(SPLIT_SEED), "| Init seed base:", int(INIT_SEED_BASE), "| Shuffle seed base:", int(SHUFFLE_SEED_BASE))  # 3.12.26 show the main seed settings
print("Split fractions:", {"train": float(TRAIN_FRAC), "val": float(VAL_FRAC), "test": float(TEST_FRAC)})  # 3.12.26 show the current split proportions
print("Training settings:", {"epochs": int(EPOCHS), "batch_size": int(BATCH_SIZE), "lr": float(LR), "num_workers": int(NUM_WORKERS), "n_splits": int(N_SPLITS), "n_init_trials": int(N_INIT_TRIALS)})  # 3.12.26 show the main training settings
print("Saving val artifact epochs:", sorted([int(e) for e in set(SAVE_VAL_ARTIFACT_EPOCHS)]))  # 3.12.26 show when validation artifacts are enabled
print("SAVE_TEST_ARTIFACTS:", bool(SAVE_TEST_ARTIFACTS), "| SAVE_TEST_MISCLASS_IMAGES:", bool(SAVE_TEST_MISCLASS_IMAGES))  # 3.12.26 show the test-artifact settings

results_summary = []  # 3.12.26 collect one summary row per group, split, and trial
all_histories = {}  # 3.12.26 collect full training histories grouped by channel group and split seed

split_seeds = [int(SPLIT_SEED) + i for i in range(int(N_SPLITS))]  # 3.12.26 exact split seeds that will be run

for split_i, split_seed in enumerate(split_seeds):  # 3.12.26 loop over balanced train/val/test splits
    train_idx_s, val_idx_s, test_idx_s, split_info = make_one_balanced_split(split_seed=int(split_seed))  # 3.12.26 build this split using the same balanced logic as Cell 3

    split_art_dir = RUN_ARTIFACTS_DIR / f"split_{int(split_seed)}"  # 3.12.26 root artifact folder for this split
    split_art_dir.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the split artifact folder exists

    split_summary_json = {  # 3.12.26 compact split summary saved once per split
        "split_seed": int(split_seed),  # 3.12.26 split seed used to build this split
        "split_index": int(split_i),  # 3.12.26 zero-based split counter
        "split_info": split_info,  # 3.12.26 detailed summary returned by make_one_balanced_split
    }  # 3.12.26 end split-summary JSON dictionary
    with open(str(split_art_dir / "split_summary.json"), "w") as f:  # 3.12.26 save split summary for reproducibility
        json.dump(split_summary_json, f, indent=2)  # 3.12.26 write the split-summary JSON file

    base_shuffle_seed_for_split = int(SHUFFLE_SEED_BASE) + 1000 * int(split_i)  # 3.12.26 use a stable shuffle-seed base for this split

    print("\n" + "#" * 90)  # 3.12.26 visual separator for split output
    print(f"SPLIT {split_i + 1}/{len(split_seeds)} | split_seed={int(split_seed)} | shuffle_seed_base={int(base_shuffle_seed_for_split)}")  # 3.12.26 print which split is running
    print("  Train/Val/Test sizes:", {"train": int(len(train_idx_s)), "val": int(len(val_idx_s)), "test": int(len(test_idx_s))})  # 3.12.26 print split sizes
    print("  Train hue counts:\n", counts_by_hue_local(train_idx_s))  # 3.12.26 print train hue counts for this split
    print("  Val hue counts:\n", counts_by_hue_local(val_idx_s))  # 3.12.26 print validation hue counts for this split
    print("  Test hue counts:\n", counts_by_hue_local(test_idx_s))  # 3.12.26 print test hue counts for this split

    for group_i, group_name in enumerate(BUILD_GROUPS):  # 3.12.26 loop over channel groups within this split
        print("\n" + "=" * 80)  # 3.12.26 visual separator for group output
        print(f"GROUP {group_i + 1}/{len(BUILD_GROUPS)}: {group_name}")  # 3.12.26 print which channel group is running

        if bool(KEEP_SHUFFLE_IDENTICAL_ACROSS_GROUPS):  # 3.12.26 optionally keep training minibatch order matched across groups
            run_shuffle_seed = int(base_shuffle_seed_for_split)  # 3.12.26 identical shuffle order across groups in this split
        else:  # 3.12.26 otherwise let each group use a different training-shuffle seed
            run_shuffle_seed = int(base_shuffle_seed_for_split) + 100 * int(group_i)  # 3.12.26 group-specific shuffle seed

        keys_subset, channel_idxs, c_run, ds_train, ds_val, ds_test, dl_train, dl_val, dl_test = make_loaders_for_group(  # 3.12.26 build split/group-specific datasets and loaders
            group_name=group_name,
            train_idx_use=train_idx_s,
            val_idx_use=val_idx_s,
            test_idx_use=test_idx_s,
            shuffle_seed=run_shuffle_seed,
        )

        print("Channels (keys):", keys_subset)  # 3.12.26 print the channel names used by this group
        print("Channels (idxs):", channel_idxs.tolist())  # 3.12.26 print the channel indices used by this group
        print("C_run:", int(c_run), "| shuffle_seed:", int(run_shuffle_seed))  # 3.12.26 print the group input-channel count and shuffle seed

        all_histories.setdefault(str(group_name), {})  # 3.12.26 ensure this group has an entry in the history dictionary
        all_histories[str(group_name)].setdefault(int(split_seed), [])  # 3.12.26 ensure this split has a history list for the current group

        criterion_split = make_criterion_for_split(train_idx_s)  # 3.12.26 build the loss function using only this split's training indices

        for trial in range(int(N_INIT_TRIALS)):  # 3.12.26 loop over repeated random initializations for this split and group
            init_seed = int(INIT_SEED_BASE) + int(trial) + 10000 * int(split_i)  # 3.12.26 distinct initialization seed for this trial
            trial_tag = f"{group_name}|split{int(split_seed)}|trial{int(trial)}|init{int(init_seed)}"  # 3.12.26 readable label for this trial

            print("\n" + "-" * 80)  # 3.12.26 visual separator for trial output
            print(f"TRIAL {trial + 1}/{int(N_INIT_TRIALS)} | {trial_tag}")  # 3.12.26 print which trial is running

            x0, _ = ds_train[0]  # 3.12.26 one real training sample used to initialize the lazy linear layer
            xb0 = x0.unsqueeze(0)  # 3.12.26 add a batch dimension for the model dry run

            model = build_seeded_model(  # 3.12.26 build a reproducibly initialized model for this group and trial
                in_channels=int(c_run),
                n_classes=int(n_classes),
                init_seed=int(init_seed),
                device=device,
                example_batch=xb0,
            )
            optimizer = make_optimizer(model, lr=float(LR))  # 3.12.26 optimizer for this trial

            trial_art_dir = split_art_dir / f"group_{group_name}" / f"trial_{int(trial)}_init_{int(init_seed)}"  # 3.12.26 artifact folder for this exact group and trial
            trial_art_dir.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the trial artifact folder exists

            ckpt_path = trial_art_dir / "best_model.pt"  # 3.12.26 best-checkpoint path for this trial

            history = train_model(  # 3.12.26 run the actual training loop for this trial
                model=model,
                dl_train=dl_train,
                dl_val=dl_val,
                optimizer=optimizer,
                criterion=criterion_split,
                epochs=int(EPOCHS),
                save_best=True,
                ckpt_path=str(ckpt_path),
                device=device,
                grad_clip=GRAD_CLIP,
                save_val_artifacts_epochs=set(SAVE_VAL_ARTIFACT_EPOCHS),
                artifacts_dir=trial_art_dir,
                max_misclass_images=MAX_MISCLASS_IMAGES_SAVE,
            )

            all_histories[str(group_name)][int(split_seed)].append(history)  # 3.12.26 store the full training history for this trial

            best_ckpt_path = Path(history["ckpt_path"])  # 3.12.26 best checkpoint saved during training
            final_ckpt_path = Path(history["final_ckpt_path"])  # 3.12.26 final checkpoint saved after the last epoch

            test_loss_best, test_acc_best, test_art_dir_best = evaluate_checkpoint_on_test(  # 3.12.26 evaluate and optionally save artifacts for the best checkpoint
                model=model,
                ckpt_path=best_ckpt_path,
                dl_test=dl_test,
                criterion=criterion_split,
                trial_art_dir=trial_art_dir,
                snapshot_name="best",
            )
            test_loss_final, test_acc_final, test_art_dir_final = evaluate_checkpoint_on_test(  # 3.12.26 evaluate and optionally save artifacts for the final checkpoint
                model=model,
                ckpt_path=final_ckpt_path,
                dl_test=dl_test,
                criterion=criterion_split,
                trial_art_dir=trial_art_dir,
                snapshot_name="final",
            )

            results_summary.append({  # 3.12.26 one compact summary row for this group, split, and trial
                "group": str(group_name),  # 3.12.26 channel group name
                "split_seed": int(split_seed),  # 3.12.26 split seed for this run
                "trial": int(trial),  # 3.12.26 zero-based trial index
                "init_seed": int(init_seed),  # 3.12.26 model-initialization seed for this trial
                "shuffle_seed": int(run_shuffle_seed),  # 3.12.26 DataLoader shuffle seed used for this run
                "n_channels": int(c_run),  # 3.12.26 number of CNN input channels in this group
                "epochs": int(EPOCHS),  # 3.12.26 number of training epochs
                "batch_size": int(BATCH_SIZE),  # 3.12.26 training batch size
                "lr": float(LR),  # 3.12.26 learning rate
                "best_epoch": int(history.get("best_epoch", -1)),  # 3.12.26 best validation epoch from training
                "best_val_acc": float(history.get("best_val_acc", np.max(history["val_acc"]))),  # 3.12.26 best validation accuracy from training
                "train_size": int(len(train_idx_s)),  # 3.12.26 train split size
                "val_size": int(len(val_idx_s)),  # 3.12.26 validation split size
                "test_size": int(len(test_idx_s)),  # 3.12.26 test split size
                "target_pool_counts": split_info["target_pool_counts"],  # 03.14.2026 save the exact weighted pool counts instead of pool_per_hue
                "test_loss_best": float(test_loss_best),  # 3.12.26 test loss at the best checkpoint
                "test_acc_best": float(test_acc_best),  # 3.12.26 test accuracy at the best checkpoint
                "test_loss_final": float(test_loss_final),  # 3.12.26 test loss at the final checkpoint
                "test_acc_final": float(test_acc_final),  # 3.12.26 test accuracy at the final checkpoint
                "checkpoint_best": str(best_ckpt_path),  # 3.12.26 file path to the best checkpoint
                "checkpoint_final": str(final_ckpt_path),  # 3.12.26 file path to the final checkpoint
                "artifacts_dir": str(trial_art_dir),  # 3.12.26 root artifact directory for this trial
                "val_artifact_dirs": history.get("val_artifact_dirs", {}),  # 3.12.26 saved validation-artifact folders from train_model
                "test_artifacts_dir_best": test_art_dir_best,  # 3.12.26 test-artifact folder for the best checkpoint
                "test_artifacts_dir_final": test_art_dir_final,  # 3.12.26 test-artifact folder for the final checkpoint
            })  # 3.12.26 end summary row for this run

df_summary = pd.DataFrame(results_summary)  # 3.12.26 convert the summary rows into one DataFrame
df_summary = df_summary.sort_values(["group", "split_seed", "trial"]).reset_index(drop=True)  # 3.12.26 keep the summary in a stable readable order

print("\n" + "=" * 80)  # 3.12.26 visual separator for the final summary
print("SUMMARY (best val acc + test_best + test_final):")  # 3.12.26 explain what the summary table contains
print(df_summary)  # 3.12.26 print the full summary table

results_csv_path = RUN_ARTIFACTS_DIR / "results_summary.csv"  # 3.12.26 output path for the saved summary table
df_summary.to_csv(str(results_csv_path), index=False)  # 3.12.26 save the summary table for later plotting and aggregation
print(f"\nSaved results_summary.csv -> {results_csv_path}")  # 3.12.26 confirm where the summary table was saved

RESULTS_SUMMARY_DF = df_summary  # 3.12.26 keep the summary DataFrame available for later notebook cells

display(df_summary)  # 3.12.26 show the summary table in the notebook output

In [ ]:
# Cell 8 — Evaluation + comparisons (CM panels per group + learning curves with TEST points)  # 3.12.26 revised to print confusion-matrix tables again and keep the berlin colormap for figures

SAVE_FIGS = True
SAVE_SVG_TOO = False
FIG_DPI = 600
FIG_OUTDIR = None
FIG_PREFIX = "cell8"

if FIG_OUTDIR is None:  # 3.12.26 default figures to the CNN experiment artifact folder instead of cwd/OUT_DIR
    FIG_OUTDIR = Path(RUN_ARTIFACTS_DIR) / "figures"  # 3.12.26 keep Cell 8 figures next to the experiment outputs
else:  # 3.12.26 preserve the option to override the figure directory manually
    FIG_OUTDIR = Path(FIG_OUTDIR)  # 3.12.26 normalize a user-supplied figure directory to a Path object

FIG_OUTDIR.mkdir(parents=True, exist_ok=True)

def _slug(s: str):
    return "".join([c if (c.isalnum() or c in ["-", "_"]) else "_" for c in str(s)])

STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

def _save_fig(fig, stem: str):
    if not SAVE_FIGS:
        return
    png_path = FIG_OUTDIR / f"{stem}.png"
    fig.savefig(png_path, dpi=int(FIG_DPI), bbox_inches="tight")
    print("Saved:", png_path)
    if SAVE_SVG_TOO:
        svg_path = FIG_OUTDIR / f"{stem}.svg"
        fig.savefig(svg_path, bbox_inches="tight")
        print("Saved:", svg_path)

assert "RESULTS_SUMMARY_DF" in globals(), "Run Cell 7 first so RESULTS_SUMMARY_DF is available."  # 3.12.26 require the summary table built by the revised Cell 7
df_summary = RESULTS_SUMMARY_DF.copy()  # 3.12.26 use the in-memory summary table from Cell 7 rather than rebuilding it here
assert len(df_summary) > 0, "RESULTS_SUMMARY_DF is empty."  # 3.12.26 fail early if no experiment results exist yet

MODEL_ID_COL = "group"
MODEL_ORDER = list(BUILD_GROUPS)

EPOCHS_USE = int(EPOCHS)

if "class_names" in globals():
    DISPLAY_LABELS = list(class_names)
elif "CLASS_ORDER" in globals():
    DISPLAY_LABELS = list(CLASS_ORDER)
else:
    DISPLAY_LABELS = [str(i) for i in range(int(n_classes))]

N_CLASSES_USE = int(n_classes) if "n_classes" in globals() else int(len(DISPLAY_LABELS))

def _verify_schema_and_labels():  # 3.12.26 simple one-time schema check before plotting
    assert DISPLAY_LABELS is not None and len(DISPLAY_LABELS) == int(N_CLASSES_USE), "DISPLAY_LABELS must match N_CLASSES_USE."  # 3.12.26 require one display label per class
    if ("class_names" in globals()) and ("CLASS_ORDER" in globals()) and (list(class_names) != list(CLASS_ORDER)):  # 3.12.26 warn if two class-label lists disagree
        print("WARNING: class_names != CLASS_ORDER")  # 3.12.26 visible warning for possible label-order mismatch
        print("  class_names:", list(class_names))  # 3.12.26 print the first label list
        print("  CLASS_ORDER:", list(CLASS_ORDER))  # 3.12.26 print the second label list
    print("Cell 8 label/schema check:")  # 3.12.26 header for the schema summary
    print("  MODEL_ID_COL:", MODEL_ID_COL)  # 3.12.26 show which summary-table column identifies the model group
    print("  N_CLASSES_USE:", int(N_CLASSES_USE))  # 3.12.26 show how many classes the plots expect
    print("  DISPLAY_LABELS:", DISPLAY_LABELS)  # 3.12.26 show the class labels used on confusion matrices
    print("  Using berlin colormap for confusion matrices.")  # 3.12.26 make the colormap choice explicit once

_verify_schema_and_labels()

CM_CMAP = "berlin"  # 3.12.26 use the berlin colormap for all confusion-matrix panels

GROUP_COLOR_MAP = {
    "Classic4": "red",
    "Classic4_plus_s": "purple",
    "Neitz4": "green",
    "Neitz8": "royalblue",
    "All9": "black",
}

def _canonical_group_name(name: str):
    s = str(name)
    low = s.lower()
    if ("classic4" in low) and (("plus" in low) or ("+s" in low) or ("plus_s" in low)):
        return "Classic4_plus_s"
    if "classic4" in low:
        return "Classic4"
    if "neitz8" in low:
        return "Neitz8"
    if "neitz4" in low:
        return "Neitz4"
    if ("all9" in low) or ("all_9" in low):
        return "All9"
    alias = {
        "Classic4+S": "Classic4_plus_s",
        "Classic4_plus_S": "Classic4_plus_s",
        "Classic4_plus_s": "Classic4_plus_s",
        "classic4": "Classic4",
        "classic4_plus_s": "Classic4_plus_s",
        "neitz4": "Neitz4",
        "neitz8": "Neitz8",
        "all9": "All9",
        "All_9": "All9",
        "All9": "All9",
    }
    return alias.get(s, s)

def _get_group_color(name: str):
    return GROUP_COLOR_MAP.get(_canonical_group_name(name), None)

TRAIN_LS = ":"
VAL_LS = "-"
VAL_LW_MULT = 1.8
TEST_MARKER = "o"

def _load_cm_npy(path: Path):  # 3.12.26 load one numeric confusion matrix safely
    try:
        cm = np.load(str(path))
        cm = np.asarray(cm, dtype=np.int64)
        if cm.shape != (int(N_CLASSES_USE), int(N_CLASSES_USE)):
            return None
        if int(cm.sum()) <= 0:
            return None
        return cm
    except Exception:
        return None

def _load_cm_from_dir(cm_dir: Path):  # 3.12.26 load the confusion matrix bundle from one artifact directory
    cm_path = Path(cm_dir) / "confusion_matrix.npy"  # 3.12.26 Cell 6 saves the numeric confusion matrix with this exact filename
    if not cm_path.exists():
        return None
    return _load_cm_npy(cm_path)

def _normalize_val_artifact_dirs(value):  # 3.12.26 normalize val_artifact_dirs so plotting code can read it safely
    if isinstance(value, dict):
        return {str(k): str(v) for k, v in value.items()}
    return {}

def _verify_bundle_files_exist(df: pd.DataFrame, n_rows: int = 2):  # 3.12.26 smoke-check that the revised Cell 6 bundle files exist where Cell 8 expects them
    if "artifacts_dir" not in df.columns:
        print("WARNING: df_summary has no 'artifacts_dir'; cannot verify confusion-matrix files.")
        return
    for k, (_, row) in enumerate(df.head(int(n_rows)).iterrows()):
        trial_dir = Path(str(row["artifacts_dir"]))
        val_dirs = _normalize_val_artifact_dirs(row.get("val_artifact_dirs", {}))  # 3.12.26 revised Cell 6 stores val artifact folders in this dict
        test_best_dir = trial_dir / "test_best"  # 3.12.26 best-checkpoint test artifacts from Cell 7
        test_final_dir = trial_dir / "test_final"  # 3.12.26 final-checkpoint test artifacts from Cell 7

        missing = []
        for snapshot_name, snapshot_dir in val_dirs.items():  # 3.12.26 check saved validation snapshot folders rather than old val_epochXX folders
            cm_path = Path(str(snapshot_dir)) / "confusion_matrix.npy"  # 3.12.26 numeric confusion matrix expected in every saved validation snapshot folder
            if not cm_path.exists():
                missing.append(str(cm_path))
        for test_dir in [test_best_dir, test_final_dir]:
            cm_path = test_dir / "confusion_matrix.npy"
            if not cm_path.exists():
                missing.append(str(cm_path))

        if len(missing) > 0:
            print(f"WARNING: missing expected confusion-matrix files for row {k}:")  # 3.12.26 visible warning without stopping the whole plotting cell
            for p in missing[:8]:
                print("  -", p)
        else:
            print(f"Confusion-matrix file check OK for row {k}.")  # 3.12.26 confirm the expected files were found

_verify_bundle_files_exist(df_summary, n_rows=2)

VAL_SNAPSHOT_ORDER = ["best", "final"]  # 3.12.26 the revised Cell 6 can save one validation snapshot with either name

def _accumulate_val_snapshot_cms(df: pd.DataFrame, model_order):  # 3.12.26 sum saved validation confusion matrices across runs by snapshot name
    val_sum = {m: {snap: np.zeros((N_CLASSES_USE, N_CLASSES_USE), dtype=np.int64) for snap in VAL_SNAPSHOT_ORDER} for m in model_order}  # 3.12.26 accumulator for validation confusion matrices
    val_n = {m: {snap: 0 for snap in VAL_SNAPSHOT_ORDER} for m in model_order}  # 3.12.26 count how many runs contributed to each validation snapshot

    for _, row in df.iterrows():
        model_name = str(row[MODEL_ID_COL])
        if model_name not in val_sum:
            continue
        val_dirs = _normalize_val_artifact_dirs(row.get("val_artifact_dirs", {}))  # 3.12.26 read validation snapshot folders recorded by Cell 6
        for snap in VAL_SNAPSHOT_ORDER:
            if snap not in val_dirs:
                continue
            cm = _load_cm_from_dir(Path(str(val_dirs[snap])))
            if cm is None:
                continue
            val_sum[model_name][snap] += cm
            val_n[model_name][snap] += 1

    return val_sum, val_n

def _accumulate_test_cms(df: pd.DataFrame, model_order):  # 3.12.26 sum test confusion matrices across runs for best and final checkpoints
    best_sum = {m: np.zeros((N_CLASSES_USE, N_CLASSES_USE), dtype=np.int64) for m in model_order}
    best_n = {m: 0 for m in model_order}
    final_sum = {m: np.zeros((N_CLASSES_USE, N_CLASSES_USE), dtype=np.int64) for m in model_order}
    final_n = {m: 0 for m in model_order}

    for _, row in df.iterrows():
        model_name = str(row[MODEL_ID_COL])
        if model_name not in best_sum:
            continue

        best_dir = row.get("test_artifacts_dir_best", None)  # 3.12.26 read the best-checkpoint test artifact directory from Cell 7
        if isinstance(best_dir, str) and len(best_dir) > 0:
            cm_best = _load_cm_from_dir(Path(best_dir))
            if cm_best is not None:
                best_sum[model_name] += cm_best
                best_n[model_name] += 1

        final_dir = row.get("test_artifacts_dir_final", None)  # 3.12.26 read the final-checkpoint test artifact directory from Cell 7
        if isinstance(final_dir, str) and len(final_dir) > 0:
            cm_final = _load_cm_from_dir(Path(final_dir))
            if cm_final is not None:
                final_sum[model_name] += cm_final
                final_n[model_name] += 1

    return best_sum, best_n, final_sum, final_n

val_cm_sum, val_cm_n = _accumulate_val_snapshot_cms(df_summary, MODEL_ORDER)  # 3.12.26 aggregate validation confusion matrices using the revised Cell 6 save scheme
test_best_cm_sum, test_best_cm_n, test_final_cm_sum, test_final_cm_n = _accumulate_test_cms(df_summary, MODEL_ORDER)

def _cm_to_df(cm, labels):  # 3.12.26 convert one confusion matrix into a labeled pandas table for text output
    return pd.DataFrame(np.asarray(cm, dtype=np.int64), index=list(labels), columns=list(labels))  # 3.12.26 build a readable table with class names on rows and columns

def _print_cm_table(cm, labels, title, n_runs):  # 3.12.26 print one confusion matrix as plain text in addition to plotting it
    print("\n" + "-" * 80)  # 3.12.26 separate the printed tables clearly in the notebook output
    print(f"{title} | n_runs={int(n_runs)}")  # 3.12.26 print a clear header for this confusion-matrix table
    print(_cm_to_df(cm, labels).to_string())  # 3.12.26 print the full labeled confusion-matrix table as text

def _plot_cm_ax(ax, cm, labels, title, vmax=None):  # 3.12.26 plot one confusion-matrix panel with the berlin colormap
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(labels))
    disp.plot(ax=ax, values_format="d", colorbar=False, cmap=CM_CMAP)  # 3.12.26 apply the berlin colormap to every confusion-matrix panel
    ax.set_title(title)
    im = ax.images[-1] if len(ax.images) > 0 else None
    if (im is not None) and (vmax is not None):
        im.set_clim(0, vmax)
    return im

for m in MODEL_ORDER:  # 3.12.26 make one confusion-matrix figure per model group
    vmax_candidates = []  # 3.12.26 collect panel maxima so all panels in this figure share one color scale
    for snap in VAL_SNAPSHOT_ORDER:
        if int(val_cm_n[m][snap]) > 0:
            vmax_candidates.append(int(np.max(val_cm_sum[m][snap])))
    if int(test_best_cm_n[m]) > 0:
        vmax_candidates.append(int(np.max(test_best_cm_sum[m])))
    if int(test_final_cm_n[m]) > 0:
        vmax_candidates.append(int(np.max(test_final_cm_sum[m])))
    vmax = int(max(vmax_candidates)) if len(vmax_candidates) > 0 else None

    print("\n" + "=" * 100)  # 3.12.26 start a new printed section for this model group
    print(f"CONFUSION MATRICES — {m}")  # 3.12.26 print the model-group header above the text tables

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))  # 3.12.26 use a simple 2x2 layout: val best, val final, test best, test final
    axes = np.asarray(axes).ravel()
    im_handles = []

    ax = axes[0]  # 3.12.26 panel 1: saved validation best snapshot
    n_runs = int(val_cm_n[m]["best"])
    if n_runs == 0:
        ax.axis("off")
        ax.text(0.5, 0.5, "VAL(best)\nNOT AVAILABLE\n(n_runs=0)", ha="center", va="center", fontsize=11)  # 3.12.26 clear message when no best validation confusion matrices were saved
    else:
        _print_cm_table(val_cm_sum[m]["best"], DISPLAY_LABELS, f"{m} | VAL(best)", n_runs)  # 3.12.26 print the best validation confusion matrix as a text table
        im = _plot_cm_ax(ax, val_cm_sum[m]["best"], DISPLAY_LABELS, title=f"VAL(best) (sum across runs; n_runs={n_runs})", vmax=vmax)  # 3.12.26 plot the summed validation-best confusion matrix
        if im is not None:
            im_handles.append(im)

    ax = axes[1]  # 3.12.26 panel 2: saved validation final snapshot
    n_runs = int(val_cm_n[m]["final"])
    if n_runs == 0:
        ax.axis("off")
        ax.text(0.5, 0.5, "VAL(final)\nNOT AVAILABLE\n(n_runs=0)", ha="center", va="center", fontsize=11)  # 3.12.26 clear message when no final validation confusion matrices were saved
    else:
        _print_cm_table(val_cm_sum[m]["final"], DISPLAY_LABELS, f"{m} | VAL(final)", n_runs)  # 3.12.26 print the final validation confusion matrix as a text table
        im = _plot_cm_ax(ax, val_cm_sum[m]["final"], DISPLAY_LABELS, title=f"VAL(final) (sum across runs; n_runs={n_runs})", vmax=vmax)  # 3.12.26 plot the summed validation-final confusion matrix
        if im is not None:
            im_handles.append(im)

    ax = axes[2]  # 3.12.26 panel 3: test best checkpoint
    n_runs = int(test_best_cm_n[m])
    if n_runs == 0:
        ax.axis("off")
        ax.text(0.5, 0.5, "TEST(best)\nNOT AVAILABLE\n(n_runs=0)", ha="center", va="center", fontsize=11)
    else:
        _print_cm_table(test_best_cm_sum[m], DISPLAY_LABELS, f"{m} | TEST(best)", n_runs)  # 3.12.26 print the best-checkpoint test confusion matrix as a text table
        im = _plot_cm_ax(ax, test_best_cm_sum[m], DISPLAY_LABELS, title=f"TEST(best) (sum across runs; n_runs={n_runs})", vmax=vmax)  # 3.12.26 plot the summed best-checkpoint test confusion matrix
        if im is not None:
            im_handles.append(im)

    ax = axes[3]  # 3.12.26 panel 4: test final checkpoint
    n_runs = int(test_final_cm_n[m])
    if n_runs == 0:
        ax.axis("off")
        ax.text(0.5, 0.5, "TEST(final)\nNOT AVAILABLE\n(n_runs=0)", ha="center", va="center", fontsize=11)
    else:
        _print_cm_table(test_final_cm_sum[m], DISPLAY_LABELS, f"{m} | TEST(final)", n_runs)  # 3.12.26 print the final-checkpoint test confusion matrix as a text table
        im = _plot_cm_ax(ax, test_final_cm_sum[m], DISPLAY_LABELS, title=f"TEST(final) (sum across runs; n_runs={n_runs})", vmax=vmax)  # 3.12.26 plot the summed final-checkpoint test confusion matrix
        if im is not None:
            im_handles.append(im)

    fig.suptitle(f"Confusion matrices — {m}", fontsize=16)  # 3.12.26 title for the model-specific confusion-matrix figure
    if (len(im_handles) > 0) and (vmax is not None):
        fig.tight_layout(rect=[0, 0.02, 0.92, 0.95])  # 3.12.26 reserve space on the right for a shared colorbar
        cax = fig.add_axes([0.94, 0.15, 0.015, 0.70])  # 3.12.26 colorbar axis on the right side of the figure
        cb = fig.colorbar(im_handles[0], cax=cax)  # 3.12.26 shared colorbar for all panels in this figure
        cb.set_label("Count")
    else:
        fig.tight_layout(rect=[0, 0.02, 1, 0.95])

    plt.show()
    _save_fig(fig, stem=f"{FIG_PREFIX}_cm_panels_{_slug(m)}_{STAMP}")

def _collect_histories(model_name: str):  # 3.12.26 collect all training histories for one model group across splits and trials
    histories = []
    obj = all_histories.get(model_name, None)
    if isinstance(obj, dict):
        for _, history_list in obj.items():
            for history in history_list:
                histories.append(history)
    elif isinstance(obj, list):
        histories = obj
    return histories

def _test_stats_from_df(model_name: str):  # 3.12.26 compute mean and std test metrics for one model group from df_summary
    mask = (df_summary[MODEL_ID_COL].astype(str) == str(model_name))
    if not bool(np.any(mask)):
        return (None,) * 9

    best_loss = df_summary.loc[mask, "test_loss_best"].values.astype(float) if "test_loss_best" in df_summary.columns else None
    best_acc = df_summary.loc[mask, "test_acc_best"].values.astype(float) if "test_acc_best" in df_summary.columns else None
    final_loss = df_summary.loc[mask, "test_loss_final"].values.astype(float) if "test_loss_final" in df_summary.columns else None
    final_acc = df_summary.loc[mask, "test_acc_final"].values.astype(float) if "test_acc_final" in df_summary.columns else None
    best_epoch = df_summary.loc[mask, "best_epoch"].values.astype(float) if "best_epoch" in df_summary.columns else None

    best_loss_mean = float(np.nanmean(best_loss)) if best_loss is not None else None
    best_loss_std = float(np.nanstd(best_loss)) if best_loss is not None else None
    best_acc_mean = float(np.nanmean(best_acc)) if best_acc is not None else None
    best_acc_std = float(np.nanstd(best_acc)) if best_acc is not None else None

    final_loss_mean = float(np.nanmean(final_loss)) if final_loss is not None else None
    final_loss_std = float(np.nanstd(final_loss)) if final_loss is not None else None
    final_acc_mean = float(np.nanmean(final_acc)) if final_acc is not None else None
    final_acc_std = float(np.nanstd(final_acc)) if final_acc is not None else None

    best_epoch_mean = float(np.nanmean(best_epoch)) if best_epoch is not None else None

    return best_loss_mean, best_loss_std, best_acc_mean, best_acc_std, final_loss_mean, final_loss_std, final_acc_mean, final_acc_std, best_epoch_mean

def _set_ylim_from_means(ax, mean_series_list, pad_frac=0.06, log=False, eps=1e-12):  # 3.12.26 keep plot limits based on means only
    vals = []
    for arr in mean_series_list:
        if arr is None:
            continue
        a = np.asarray(arr, dtype=float).ravel()
        a = a[np.isfinite(a)]
        vals.extend(a.tolist())
    if len(vals) == 0:
        return
    vmin = float(np.min(vals))
    vmax = float(np.max(vals))

    if log:
        vmin = max(vmin, eps)
        vmax = max(vmax, eps)
        lo = max(vmin / (1.0 + pad_frac), eps)
        hi = max(vmax * (1.0 + pad_frac), lo * 1.01)
        ax.set_ylim(lo, hi)
    else:
        rng = max(vmax - vmin, eps)
        lo = vmin - pad_frac * rng
        hi = vmax + pad_frac * rng
        if hi <= lo:
            hi = lo + eps
        ax.set_ylim(lo, hi)

EPS_LOSS = 1e-12

fig, ax = plt.subplots(figsize=(10.5, 4.4))
means_for_ylim = []

for m in MODEL_ORDER:
    hs = _collect_histories(m)
    if len(hs) == 0:
        continue

    color = _get_group_color(m)

    L = min(len(h["train_loss"]) for h in hs)
    e = np.arange(1, L + 1)

    tr = np.vstack([np.asarray(h["train_loss"][:L], dtype=float) for h in hs])
    va = np.vstack([np.asarray(h["val_loss"][:L], dtype=float) for h in hs])

    tr_m, tr_s = tr.mean(axis=0), tr.std(axis=0)
    va_m, va_s = va.mean(axis=0), va.std(axis=0)

    tr_m_plot = np.maximum(tr_m, EPS_LOSS)
    va_m_plot = np.maximum(va_m, EPS_LOSS)

    tr_lo = np.maximum(tr_m - tr_s, EPS_LOSS)
    tr_hi = np.maximum(tr_m + tr_s, EPS_LOSS)
    va_lo = np.maximum(va_m - va_s, EPS_LOSS)
    va_hi = np.maximum(va_m + va_s, EPS_LOSS)

    ax.plot(e, tr_m_plot, linestyle=TRAIN_LS, color=color, label=f"{m} train (mean)")
    ax.fill_between(e, tr_lo, tr_hi, alpha=0.15, color=color)

    ax.plot(e, va_m_plot, linestyle=VAL_LS, linewidth=plt.rcParams["lines.linewidth"] * VAL_LW_MULT, color=color, label=f"{m} val (mean)")
    ax.fill_between(e, va_lo, va_hi, alpha=0.15, color=color)

    means_for_ylim.append(tr_m_plot)
    means_for_ylim.append(va_m_plot)

    bl_m, bl_s, _, _, fl_m, fl_s, _, _, be_m = _test_stats_from_df(m)

    if (bl_m is not None) and np.isfinite(bl_m):
        x_best = float(be_m) if (be_m is not None and np.isfinite(be_m)) else float(L)
        y_best = float(max(bl_m, EPS_LOSS))
        yerr_best = float(0.0 if (bl_s is None or (not np.isfinite(bl_s))) else bl_s)
        lo = max(y_best - yerr_best, EPS_LOSS)
        hi = max(y_best + yerr_best, EPS_LOSS)
        yerr = np.array([[y_best - lo], [hi - y_best]], dtype=float)
        ax.errorbar([x_best], [y_best], yerr=yerr, capsize=3, fmt=TEST_MARKER, color=color, markeredgecolor=color, markerfacecolor="none", label=f"{m} TEST(best) (mean±std)")  # 3.12.26 keep the best-checkpoint test point as an open circle

    if (fl_m is not None) and np.isfinite(fl_m):
        x_final = float(EPOCHS_USE)
        y_final = float(max(fl_m, EPS_LOSS))
        yerr_final = float(0.0 if (fl_s is None or (not np.isfinite(fl_s))) else fl_s)
        lo = max(y_final - yerr_final, EPS_LOSS)
        hi = max(y_final + yerr_final, EPS_LOSS)
        yerr = np.array([[y_final - lo], [hi - y_final]], dtype=float)
        ax.errorbar([x_final], [y_final], yerr=yerr, capsize=3, fmt=TEST_MARKER, color=color, markeredgecolor=color, markerfacecolor=color, label=f"{m} TEST(final) (mean±std)")  # 3.12.26 keep the final-checkpoint test point as a filled circle

    if (bl_m is not None) and np.isfinite(bl_m):
        means_for_ylim.append([float(max(bl_m, EPS_LOSS))])
    if (fl_m is not None) and np.isfinite(fl_m):
        means_for_ylim.append([float(max(fl_m, EPS_LOSS))])

ax.set_yscale("log")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Learning curves — Loss (train/val per epoch; TEST(best) + TEST(final) shown as points)")

_set_ylim_from_means(ax, means_for_ylim, pad_frac=0.08, log=True, eps=EPS_LOSS)

ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0)
fig.subplots_adjust(right=0.72)
fig.tight_layout()
plt.show()

_save_fig(fig, stem=f"{FIG_PREFIX}_learning_curves_loss_with_test_best_final_{STAMP}")

fig, ax = plt.subplots(figsize=(10.5, 4.4))
means_for_ylim = []

for m in MODEL_ORDER:
    hs = _collect_histories(m)
    if len(hs) == 0:
        continue

    color = _get_group_color(m)

    L = min(len(h["train_acc"]) for h in hs)
    e = np.arange(1, L + 1)

    tr = np.vstack([np.asarray(h["train_acc"][:L], dtype=float) for h in hs])
    va = np.vstack([np.asarray(h["val_acc"][:L], dtype=float) for h in hs])

    tr_m, tr_s = tr.mean(axis=0), tr.std(axis=0)
    va_m, va_s = va.mean(axis=0), va.std(axis=0)

    ax.plot(e, tr_m, linestyle=TRAIN_LS, color=color, label=f"{m} train (mean)")
    ax.fill_between(e, tr_m - tr_s, tr_m + tr_s, alpha=0.15, color=color)

    ax.plot(e, va_m, linestyle=VAL_LS, linewidth=plt.rcParams["lines.linewidth"] * VAL_LW_MULT, color=color, label=f"{m} val (mean)")
    ax.fill_between(e, va_m - va_s, va_m + va_s, alpha=0.15, color=color)

    means_for_ylim.append(tr_m)
    means_for_ylim.append(va_m)

    _, _, ba_m, ba_s, _, _, fa_m, fa_s, be_m = _test_stats_from_df(m)

    if (ba_m is not None) and np.isfinite(ba_m):
        x_best = float(be_m) if (be_m is not None and np.isfinite(be_m)) else float(L)
        y_best = float(ba_m)
        yerr_best = float(0.0 if (ba_s is None or (not np.isfinite(ba_s))) else ba_s)
        ax.errorbar([x_best], [y_best], yerr=[yerr_best], capsize=3, fmt=TEST_MARKER, color=color, markeredgecolor=color, markerfacecolor="none", label=f"{m} TEST(best) (mean±std)")  # 3.12.26 keep the best-checkpoint test point as an open circle

    if (fa_m is not None) and np.isfinite(fa_m):
        x_final = float(EPOCHS_USE)
        y_final = float(fa_m)
        yerr_final = float(0.0 if (fa_s is None or (not np.isfinite(fa_s))) else fa_s)
        ax.errorbar([x_final], [y_final], yerr=[yerr_final], capsize=3, fmt=TEST_MARKER, color=color, markeredgecolor=color, markerfacecolor=color, label=f"{m} TEST(final) (mean±std)")  # 3.12.26 keep the final-checkpoint test point as a filled circle

    if (ba_m is not None) and np.isfinite(ba_m):
        means_for_ylim.append([float(ba_m)])
    if (fa_m is not None) and np.isfinite(fa_m):
        means_for_ylim.append([float(fa_m)])

ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Learning curves — Accuracy (train/val per epoch; TEST(best) + TEST(final) shown as points)")

_set_ylim_from_means(ax, means_for_ylim, pad_frac=0.05, log=False)

ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0)
fig.subplots_adjust(right=0.72)
fig.tight_layout()
plt.show()

_save_fig(fig, stem=f"{FIG_PREFIX}_learning_curves_accuracy_with_test_best_final_{STAMP}")

print("Figure output directory:", FIG_OUTDIR)

In [ ]:
# Cell 9 — Save evaluation outputs (CSV-focused snapshot)  # 3.12.26 simplified: save the main outputs in CSV form wherever possible

assert "RESULTS_SUMMARY_DF" in globals(), "RESULTS_SUMMARY_DF not found. Run Cell 7 first."  # 3.12.26 require the summary table built by Cell 7
assert "class_names" in globals(), "class_names not found. Run Cell 3 first."  # 3.12.26 require the class labels before saving label-dependent outputs

df_summary = RESULTS_SUMMARY_DF.copy()  # 3.12.26 use the experiment summary from Cell 7 as the main table for this snapshot
df_eval = pd.DataFrame(eval_rows) if "eval_rows" in globals() else pd.DataFrame()  # 3.12.26 keep compatibility with any optional evaluation rows without requiring them

HAS_VAL_CMS = ("val_cm_sum" in globals()) and ("val_cm_n" in globals())  # 3.12.26 detect whether Cell 8 built aggregated validation confusion matrices
HAS_TEST_BEST_CMS = ("test_best_cm_sum" in globals()) and ("test_best_cm_n" in globals())  # 3.12.26 detect whether best-checkpoint test confusion matrices exist
HAS_TEST_FINAL_CMS = ("test_final_cm_sum" in globals()) and ("test_final_cm_n" in globals())  # 3.12.26 detect whether final-checkpoint test confusion matrices exist
HAS_REPORTS = ("reports" in globals())  # 3.12.26 detect whether legacy classification reports exist
HAS_HISTORIES = ("all_histories" in globals())  # 3.12.26 detect whether full training histories exist

assert HAS_VAL_CMS or HAS_TEST_BEST_CMS or HAS_TEST_FINAL_CMS or HAS_REPORTS or HAS_HISTORIES, "No evaluation outputs found to save."  # 3.12.26 require at least one output type before making a snapshot

def _safe_name(text):  # 3.12.26 simple filename-safe helper that does not need extra imports
    return "".join([c if (c.isalnum() or c in ["-", "_", "."]) else "_" for c in str(text)]).strip("_")  # 3.12.26 keep only simple filename characters

def _pick_base_dir():  # 3.12.26 choose the snapshot root near the exported dataset whenever possible
    if ("EXPORT_DIR" in globals()) and (EXPORT_DIR is not None) and (str(EXPORT_DIR).strip() != ""):  # 3.12.26 prefer the export directory from Cell 1
        p = Path(str(EXPORT_DIR)).expanduser().resolve()  # 3.12.26 normalize EXPORT_DIR to an absolute Path
        if p.name.endswith(".zarr"):  # 3.12.26 tolerate being given the zarr store path by mistake
            return p.parent  # 3.12.26 use the parent folder when EXPORT_DIR points to dataset.zarr
        return p  # 3.12.26 use EXPORT_DIR directly when it is already the run folder
    if "RUN_ARTIFACTS_DIR" in globals():  # 3.12.26 otherwise keep the snapshot near the CNN artifacts folder
        return Path(str(RUN_ARTIFACTS_DIR)).expanduser().resolve().parent  # 3.12.26 use the parent of the artifacts folder as the base directory
    return Path.cwd()  # 3.12.26 final fallback is the current working directory

BASE_DATA_DIR = _pick_base_dir()  # 3.12.26 base folder where the snapshot output directory will be created
RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")  # 3.12.26 timestamped folder name without needing a datetime import
OUT_DIR = BASE_DATA_DIR / f"cnn_eval_outputs_{RUN_TIMESTAMP}"  # 3.12.26 root folder for this saved snapshot
OUT_DIR.mkdir(parents=True, exist_ok=True)  # 3.12.26 create the snapshot folder before writing files

print("Saving outputs to:", OUT_DIR)  # 3.12.26 show where all files from this cell will be written

tables_dir = OUT_DIR / "tables"  # 3.12.26 folder for summary tables
tables_dir.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the tables folder exists

train_csv = tables_dir / "cnn_train_summary.csv"  # 3.12.26 main summary table from Cell 7
df_summary.to_csv(train_csv, index=False)  # 3.12.26 save the main training and test summary table as CSV
print("Saved train summary CSV:", train_csv)  # 3.12.26 confirm the training summary save

eval_csv = tables_dir / "cnn_eval_summary.csv"  # 3.12.26 optional secondary evaluation table
df_eval.to_csv(eval_csv, index=False)  # 3.12.26 save the optional evaluation table even if it is empty
print("Saved eval summary CSV:", eval_csv)  # 3.12.26 confirm the evaluation summary save

run_config_rows = []  # 3.12.26 collect one-row configuration tables as simple key-value pairs
run_config_rows.append({"setting": "timestamp", "value": str(RUN_TIMESTAMP)})  # 3.12.26 snapshot timestamp
run_config_rows.append({"setting": "out_dir", "value": str(OUT_DIR)})  # 3.12.26 snapshot output folder
run_config_rows.append({"setting": "base_data_dir", "value": str(BASE_DATA_DIR)})  # 3.12.26 base data folder used for the snapshot
run_config_rows.append({"setting": "export_dir", "value": str(EXPORT_DIR) if "EXPORT_DIR" in globals() else ""})  # 3.12.26 export folder from Cell 1 when available
run_config_rows.append({"setting": "device", "value": str(device) if "device" in globals() else "unknown"})  # 3.12.26 compute device used by the notebook
run_config_rows.append({"setting": "n_classes", "value": int(len(class_names))})  # 3.12.26 number of classes in the current label system
run_config_rows.append({"setting": "class_names", "value": "|".join([str(x) for x in class_names])})  # 3.12.26 class labels in one pipe-separated string
run_config_rows.append({"setting": "model_order", "value": "|".join([str(x) for x in MODEL_ORDER]) if "MODEL_ORDER" in globals() else ""})  # 3.12.26 model plotting order when available
run_config_rows.append({"setting": "display_labels", "value": "|".join([str(x) for x in DISPLAY_LABELS]) if "DISPLAY_LABELS" in globals() else ""})  # 3.12.26 confusion-matrix labels when available
run_config_rows.append({"setting": "epochs", "value": int(EPOCHS) if "EPOCHS" in globals() else ""})  # 3.12.26 number of training epochs
run_config_rows.append({"setting": "epochs_use", "value": int(EPOCHS_USE) if "EPOCHS_USE" in globals() else ""})  # 3.12.26 number of epochs used by Cell 8
run_config_rows.append({"setting": "lr", "value": float(LR) if "LR" in globals() else ""})  # 3.12.26 learning rate
run_config_rows.append({"setting": "batch_size", "value": int(BATCH_SIZE) if "BATCH_SIZE" in globals() else ""})  # 3.12.26 batch size
run_config_rows.append({"setting": "n_splits", "value": int(N_SPLITS) if "N_SPLITS" in globals() else ""})  # 3.12.26 number of split seeds run
run_config_rows.append({"setting": "n_init_trials", "value": int(N_INIT_TRIALS) if "N_INIT_TRIALS" in globals() else ""})  # 3.12.26 number of repeated initializations per split
run_config_rows.append({"setting": "split_seed_base", "value": int(SPLIT_SEED) if "SPLIT_SEED" in globals() else ""})  # 3.12.26 base split seed
run_config_rows.append({"setting": "init_seed_base", "value": int(INIT_SEED_BASE) if "INIT_SEED_BASE" in globals() else ""})  # 3.12.26 base initialization seed
run_config_rows.append({"setting": "shuffle_seed_base", "value": int(SHUFFLE_SEED_BASE) if "SHUFFLE_SEED_BASE" in globals() else ""})  # 3.12.26 base DataLoader shuffle seed
run_config_rows.append({"setting": "train_frac", "value": float(TRAIN_FRAC) if "TRAIN_FRAC" in globals() else ""})  # 3.12.26 train split fraction
run_config_rows.append({"setting": "val_frac", "value": float(VAL_FRAC) if "VAL_FRAC" in globals() else ""})  # 3.12.26 validation split fraction
run_config_rows.append({"setting": "test_frac", "value": float(TEST_FRAC) if "TEST_FRAC" in globals() else ""})  # 3.12.26 test split fraction
run_config_rows.append({"setting": "torch_version", "value": str(torch.__version__)})  # 3.12.26 PyTorch version used in this notebook
run_config_rows.append({"setting": "cuda_available", "value": bool(torch.cuda.is_available())})  # 3.12.26 whether CUDA was available
run_config_rows.append({"setting": "pandas_version", "value": str(pd.__version__)})  # 3.12.26 pandas version
run_config_rows.append({"setting": "numpy_version", "value": str(np.__version__)})  # 3.12.26 NumPy version
run_config_rows.append({"setting": "matplotlib_version", "value": str(plt.matplotlib.__version__)})  # 3.12.26 matplotlib version from pyplot
run_config_rows.append({"setting": "python_version", "value": str(os.sys.version)})  # 3.12.26 Python version from the current environment

run_config_csv = OUT_DIR / "run_config.csv"  # 3.12.26 save run settings as CSV instead of JSON
pd.DataFrame(run_config_rows).to_csv(run_config_csv, index=False)  # 3.12.26 write the run settings in an easy-to-read key-value CSV
print("Saved run config CSV:", run_config_csv)  # 3.12.26 confirm the run-config save

splits_dir = OUT_DIR / "splits"  # 3.12.26 folder for split indices and split summaries
splits_dir.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the splits folder exists

def _resolve_cond_masks():  # 3.12.26 read optional condition masks if they were defined earlier in the notebook
    if ("COND" in globals()) and isinstance(COND, dict) and ("mask" in COND) and isinstance(COND["mask"], dict):  # 3.12.26 prefer the revised Cell 2b COND structure
        return {str(k): np.asarray(v, dtype=bool).ravel() for k, v in COND["mask"].items()}  # 3.12.26 return the condition masks as a simple name-to-bool-array dictionary
    if ("COND_MASKS" in globals()) and isinstance(COND_MASKS, dict):  # 3.12.26 allow an older explicit COND_MASKS dictionary too
        return {str(k): np.asarray(v, dtype=bool).ravel() for k, v in COND_MASKS.items()}  # 3.12.26 normalize those masks to a simple dictionary
    return {}  # 3.12.26 return an empty dictionary when no condition masks are available

def _save_split_outputs():  # 3.12.26 rebuild the saved split indices deterministically and write them as CSV tables
    if "make_one_balanced_split" not in globals():  # 3.12.26 stop early if the revised Cell 7 split function is not available
        print("Note: make_one_balanced_split not found — skipping split export.")  # 3.12.26 explain why split files are being skipped
        return

    split_seeds = [int(SPLIT_SEED) + i for i in range(int(N_SPLITS))]  # 3.12.26 exact split seeds used by Cell 7
    split_meta_rows = []  # 3.12.26 one row per split with overall counts
    split_index_rows = []  # 3.12.26 long-format table of every saved train, val, and test index
    split_hue_rows = []  # 3.12.26 long-format hue-count table for every split and subset
    split_balance_rows = []  # 3.12.26 long-format balance-bin count table for every split and subset
    split_cond_rows = []  # 3.12.26 long-format optional condition-mask summary table

    cond_masks = _resolve_cond_masks()  # 3.12.26 read any optional condition masks now so they can be summarized per split

    for split_seed in split_seeds:  # 3.12.26 rebuild every split that Cell 7 used
        train_idx_s, val_idx_s, test_idx_s, split_info = make_one_balanced_split(split_seed=int(split_seed))  # 3.12.26 deterministically recreate the saved split indices and split summary
        split_meta_rows.append({  # 3.12.26 one compact summary row for this split
            "split_seed": int(split_seed),  # 3.12.26 split seed used to build this split
            "target_class_weights": str(split_info["target_class_weights"]),  # 03.15.2026 save the weighted class targets used by the revised split logic
            "target_pool_counts": str(split_info["target_pool_counts"]),  # 03.15.2026 save exact weighted pool counts instead of the old pool_per_hue field
            "pool_size": int(split_info["pool_size"]),  # 3.12.26 total balanced-pool size
            "train_size": int(split_info["train_size"]),  # 3.12.26 train subset size
            "val_size": int(split_info["val_size"]),  # 3.12.26 validation subset size
            "test_size": int(split_info["test_size"]),  # 3.12.26 test subset size
            "n_unique_train_bins": int(split_info["n_unique_train_bins"]),  # 3.12.26 number of unique hue-plus-condition bins in train
            "n_unique_val_bins": int(split_info["n_unique_val_bins"]),  # 3.12.26 number of unique hue-plus-condition bins in validation
            "n_unique_test_bins": int(split_info["n_unique_test_bins"]),  # 3.12.26 number of unique hue-plus-condition bins in test
        })

        for subset_name, idxs in [("train", train_idx_s), ("val", val_idx_s), ("test", test_idx_s)]:  # 3.12.26 process train, validation, and test indices in the same way
            idxs = np.asarray(idxs, dtype=np.int64).ravel()  # 3.12.26 normalize the current subset indices to one integer array

            for position_in_subset, stim_id in enumerate(idxs):  # 3.12.26 write every saved stimulus index in long format so the split can be reconstructed from CSV
                split_index_rows.append({  # 3.12.26 one long-format row for one saved sample
                    "split_seed": int(split_seed),  # 3.12.26 split seed for this row
                    "subset": str(subset_name),  # 3.12.26 train, val, or test
                    "position_in_subset": int(position_in_subset),  # 3.12.26 row order inside the subset
                    "stimulus_index": int(stim_id),  # 3.12.26 global stimulus index from the dataset
                })

            sub_meta = meta.iloc[idxs].copy()  # 3.12.26 metadata rows for this split subset
            hue_counts = sub_meta["hue"].astype(str).value_counts().reindex(CLASS_ORDER).fillna(0).astype(int)  # 3.12.26 hue counts in a stable class order
            for hue_name, count in hue_counts.items():  # 3.12.26 save hue counts as long-format rows
                split_hue_rows.append({  # 3.12.26 one hue-count row for this split subset
                    "split_seed": int(split_seed),  # 3.12.26 split seed for this row
                    "subset": str(subset_name),  # 3.12.26 train, val, or test
                    "hue": str(hue_name),  # 3.12.26 hue label
                    "count": int(count),  # 3.12.26 number of samples with this hue
                    "fraction": float(int(count) / max(1, len(idxs))),  # 3.12.26 fraction of the subset with this hue
                })

            if "balance_bin" in sub_meta.columns:  # 3.12.26 save the balance-bin composition used by the revised Cell 3
                balance_counts = sub_meta["balance_bin"].astype(str).value_counts()  # 3.12.26 count the combined intensity-saturation-background bins for this subset
                for balance_bin_name, count in balance_counts.items():  # 3.12.26 save the balance-bin counts in long format
                    split_balance_rows.append({  # 3.12.26 one balance-bin row for this split subset
                        "split_seed": int(split_seed),  # 3.12.26 split seed for this row
                        "subset": str(subset_name),  # 3.12.26 train, val, or test
                        "balance_bin": str(balance_bin_name),  # 3.12.26 combined balance-bin label from Cell 3
                        "count": int(count),  # 3.12.26 number of samples in this bin
                        "fraction": float(int(count) / max(1, len(idxs))),  # 3.12.26 fraction of the subset in this bin
                    })

            for cond_name, cond_mask in cond_masks.items():  # 3.12.26 optionally summarize any saved condition masks over this split subset
                if int(len(cond_mask)) != int(len(meta)):  # 3.12.26 skip malformed masks whose length does not match the metadata table
                    continue  # 3.12.26 do not save rows for a malformed condition mask
                n_true = int(np.sum(cond_mask[idxs]))  # 3.12.26 number of subset samples where the condition mask is true
                split_cond_rows.append({  # 3.12.26 one condition-summary row for this split subset
                    "split_seed": int(split_seed),  # 3.12.26 split seed for this row
                    "subset": str(subset_name),  # 3.12.26 train, val, or test
                    "condition": str(cond_name),  # 3.12.26 condition-mask name
                    "n_true": int(n_true),  # 3.12.26 number of true entries in this subset
                    "fraction_true": float(int(n_true) / max(1, len(idxs))),  # 3.12.26 fraction of the subset where this condition is true
                })

    pd.DataFrame(split_meta_rows).to_csv(splits_dir / "split_meta.csv", index=False)  # 3.12.26 save one compact summary row per split
    pd.DataFrame(split_index_rows).to_csv(splits_dir / "split_indices.csv", index=False)  # 3.12.26 save all train, validation, and test indices in one long-format CSV
    pd.DataFrame(split_hue_rows).to_csv(splits_dir / "split_hue_counts.csv", index=False)  # 3.12.26 save split hue counts as CSV
    pd.DataFrame(split_balance_rows).to_csv(splits_dir / "split_balance_bin_counts.csv", index=False)  # 3.12.26 save split balance-bin counts as CSV
    if len(split_cond_rows) > 0:  # 3.12.26 write the optional condition summary only when there is something to save
        pd.DataFrame(split_cond_rows).to_csv(splits_dir / "split_condition_mask_summary.csv", index=False)  # 3.12.26 save condition-mask summaries as CSV

    print("Saved split CSV files under:", splits_dir)  # 3.12.26 confirm that the split tables were written

_save_split_outputs()  # 3.12.26 run the split export now so the saved snapshot includes the train, validation, and test composition

mis_dir = OUT_DIR / "misclassified"  # 3.12.26 folder for misclassified-example records when present
mis_dir.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the misclassified folder exists

df_mis = df_misclassified.copy() if "df_misclassified" in globals() else (pd.DataFrame(misclassified_rows) if "misclassified_rows" in globals() else None)  # 3.12.26 support either a DataFrame or a list of rows for misclassified examples
if df_mis is not None:  # 3.12.26 save the misclassified table only when it exists
    mis_csv = mis_dir / "misclassified_records.csv"  # 3.12.26 file path for the misclassified records CSV
    df_mis.to_csv(mis_csv, index=False)  # 3.12.26 write the misclassified records as CSV
    print("Saved misclassified records CSV:", mis_csv)  # 3.12.26 confirm the misclassified-record save
else:
    print("Note: no misclassified rows found — skipping misclassified CSV export.")  # 3.12.26 explain why the misclassified folder may stay empty

hist_dir = OUT_DIR / "histories"  # 3.12.26 folder for per-epoch training histories
hist_dir.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the histories folder exists

if HAS_HISTORIES:  # 3.12.26 save history tables only when all_histories exists
    history_run_rows = []  # 3.12.26 one row per run summarizing the saved history
    history_epoch_rows = []  # 3.12.26 one row per epoch for every model, split, and trial

    for model_name, split_dict in all_histories.items():  # 3.12.26 walk through the nested history dictionary built by Cell 7
        for split_seed, history_list in split_dict.items():  # 3.12.26 walk through each split seed for this model
            for trial_idx, history in enumerate(history_list):  # 3.12.26 walk through each trial history within this split
                history_run_rows.append({  # 3.12.26 compact run-level summary row
                    "group": str(model_name),  # 3.12.26 model group name
                    "split_seed": int(split_seed),  # 3.12.26 split seed for this history
                    "trial_index": int(trial_idx),  # 3.12.26 zero-based trial index in the saved history list
                    "best_epoch": int(history.get("best_epoch", -1)),  # 3.12.26 best validation epoch recorded during training
                    "best_val_acc": float(history.get("best_val_acc", np.nan)),  # 3.12.26 best validation accuracy recorded during training
                    "checkpoint_best": str(history.get("ckpt_path", "")),  # 3.12.26 saved path to the best checkpoint
                    "checkpoint_final": str(history.get("final_ckpt_path", "")),  # 3.12.26 saved path to the final checkpoint
                    "val_artifact_dir_best": str(history.get("val_artifact_dirs", {}).get("best", "")),  # 3.12.26 saved best-validation artifact folder when present
                    "val_artifact_dir_final": str(history.get("val_artifact_dirs", {}).get("final", "")),  # 3.12.26 saved final-validation artifact folder when present
                })

                n_epochs_hist = int(len(history.get("train_loss", [])))  # 3.12.26 number of epochs recorded in this history
                for epoch_i in range(n_epochs_hist):  # 3.12.26 write a long-format row for every saved epoch
                    history_epoch_rows.append({  # 3.12.26 one per-epoch history row
                        "group": str(model_name),  # 3.12.26 model group name
                        "split_seed": int(split_seed),  # 3.12.26 split seed for this history row
                        "trial_index": int(trial_idx),  # 3.12.26 zero-based trial index
                        "epoch": int(epoch_i + 1),  # 3.12.26 one-based epoch number
                        "train_loss": float(history["train_loss"][epoch_i]),  # 3.12.26 training loss at this epoch
                        "train_acc": float(history["train_acc"][epoch_i]),  # 3.12.26 training accuracy at this epoch
                        "val_loss": float(history["val_loss"][epoch_i]),  # 3.12.26 validation loss at this epoch
                        "val_acc": float(history["val_acc"][epoch_i]),  # 3.12.26 validation accuracy at this epoch
                        "epoch_sec": float(history["epoch_sec"][epoch_i]),  # 3.12.26 elapsed seconds for this epoch
                    })

    history_runs_csv = hist_dir / "all_histories_runs.csv"  # 3.12.26 compact run-level history summary
    history_epochs_csv = hist_dir / "all_histories_epochs.csv"  # 3.12.26 long-format per-epoch history table
    pd.DataFrame(history_run_rows).to_csv(history_runs_csv, index=False)  # 3.12.26 save run-level history summaries as CSV
    pd.DataFrame(history_epoch_rows).to_csv(history_epochs_csv, index=False)  # 3.12.26 save per-epoch histories as CSV
    print("Saved history CSV files under:", hist_dir)  # 3.12.26 confirm the history-table saves
else:
    print("Note: all_histories not found — skipping history export.")  # 3.12.26 explain why no history files were written

cm_dir = OUT_DIR / "confusion_matrices"  # 3.12.26 folder for aggregated confusion-matrix tables
cm_dir.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the confusion-matrix folder exists

cm_manifest_rows = []  # 3.12.26 one row per saved confusion matrix so it is easy to find the files later

if HAS_VAL_CMS:  # 3.12.26 save aggregated validation confusion matrices from Cell 8
    val_dir = cm_dir / "val"  # 3.12.26 folder for validation confusion matrices
    val_dir.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the validation confusion-matrix folder exists

    for model_name in MODEL_ORDER:  # 3.12.26 save one or two validation confusion matrices per model group
        for snapshot_name in ["best", "final"]:  # 3.12.26 revised Cell 6 saves validation snapshots under these names
            if model_name not in val_cm_sum or snapshot_name not in val_cm_sum[model_name]:  # 3.12.26 skip snapshots that do not exist in the current run
                continue
            cm = np.asarray(val_cm_sum[model_name][snapshot_name], dtype=np.int64)  # 3.12.26 aggregated confusion matrix for this model and validation snapshot
            n_runs = int(val_cm_n[model_name][snapshot_name])  # 3.12.26 number of runs contributing to this aggregated validation confusion matrix

            per_model_dir = val_dir / _safe_name(model_name)  # 3.12.26 model-specific folder under the validation confusion-matrix directory
            per_model_dir.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the model-specific validation folder exists

            cm_csv = per_model_dir / f"val_{snapshot_name}_cm_sum.csv"  # 3.12.26 CSV file for the aggregated validation confusion matrix
            meta_csv = per_model_dir / f"val_{snapshot_name}_meta.csv"  # 3.12.26 CSV metadata file for the validation confusion matrix
            pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(cm_csv)  # 3.12.26 save the confusion matrix counts with class labels on rows and columns
            pd.DataFrame([{"group": str(model_name), "split": "val", "snapshot": str(snapshot_name), "n_runs": int(n_runs), "class_names": "|".join([str(x) for x in class_names])}]).to_csv(meta_csv, index=False)  # 3.12.26 save the validation confusion-matrix metadata as a simple one-row CSV

            cm_manifest_rows.append({  # 3.12.26 add this saved validation confusion matrix to the manifest table
                "group": str(model_name),  # 3.12.26 model group name
                "split": "val",  # 3.12.26 evaluation split name
                "snapshot": str(snapshot_name),  # 3.12.26 best or final validation snapshot
                "n_runs": int(n_runs),  # 3.12.26 number of contributing runs
                "cm_csv": str(cm_csv),  # 3.12.26 path to the confusion-matrix CSV
                "meta_csv": str(meta_csv),  # 3.12.26 path to the metadata CSV
            })

if HAS_TEST_BEST_CMS:  # 3.12.26 save aggregated best-checkpoint test confusion matrices from Cell 8
    test_best_dir = cm_dir / "test_best"  # 3.12.26 folder for best-checkpoint test confusion matrices
    test_best_dir.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the best-checkpoint test folder exists

    for model_name in MODEL_ORDER:  # 3.12.26 save one best-checkpoint test confusion matrix per model group
        if model_name not in test_best_cm_sum:  # 3.12.26 skip models that do not exist in the current run
            continue
        cm = np.asarray(test_best_cm_sum[model_name], dtype=np.int64)  # 3.12.26 aggregated best-checkpoint test confusion matrix
        n_runs = int(test_best_cm_n[model_name])  # 3.12.26 number of runs contributing to this aggregated best-checkpoint test confusion matrix

        per_model_dir = test_best_dir / _safe_name(model_name)  # 3.12.26 model-specific folder under the best-checkpoint test directory
        per_model_dir.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the model-specific best-checkpoint test folder exists

        cm_csv = per_model_dir / "test_best_cm_sum.csv"  # 3.12.26 CSV file for the aggregated best-checkpoint test confusion matrix
        meta_csv = per_model_dir / "test_best_meta.csv"  # 3.12.26 CSV metadata file for the best-checkpoint test confusion matrix
        pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(cm_csv)  # 3.12.26 save the best-checkpoint test confusion matrix as labeled CSV
        pd.DataFrame([{"group": str(model_name), "split": "test", "snapshot": "best", "n_runs": int(n_runs), "class_names": "|".join([str(x) for x in class_names])}]).to_csv(meta_csv, index=False)  # 3.12.26 save the best-checkpoint test confusion-matrix metadata as CSV

        cm_manifest_rows.append({  # 3.12.26 add this saved best-checkpoint test confusion matrix to the manifest table
            "group": str(model_name),  # 3.12.26 model group name
            "split": "test",  # 3.12.26 evaluation split name
            "snapshot": "best",  # 3.12.26 best checkpoint
            "n_runs": int(n_runs),  # 3.12.26 number of contributing runs
            "cm_csv": str(cm_csv),  # 3.12.26 path to the confusion-matrix CSV
            "meta_csv": str(meta_csv),  # 3.12.26 path to the metadata CSV
        })

if HAS_TEST_FINAL_CMS:  # 3.12.26 save aggregated final-checkpoint test confusion matrices from Cell 8
    test_final_dir = cm_dir / "test_final"  # 3.12.26 folder for final-checkpoint test confusion matrices
    test_final_dir.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the final-checkpoint test folder exists

    for model_name in MODEL_ORDER:  # 3.12.26 save one final-checkpoint test confusion matrix per model group
        if model_name not in test_final_cm_sum:  # 3.12.26 skip models that do not exist in the current run
            continue
        cm = np.asarray(test_final_cm_sum[model_name], dtype=np.int64)  # 3.12.26 aggregated final-checkpoint test confusion matrix
        n_runs = int(test_final_cm_n[model_name])  # 3.12.26 number of runs contributing to this aggregated final-checkpoint test confusion matrix

        per_model_dir = test_final_dir / _safe_name(model_name)  # 3.12.26 model-specific folder under the final-checkpoint test directory
        per_model_dir.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the model-specific final-checkpoint test folder exists

        cm_csv = per_model_dir / "test_final_cm_sum.csv"  # 3.12.26 CSV file for the aggregated final-checkpoint test confusion matrix
        meta_csv = per_model_dir / "test_final_meta.csv"  # 3.12.26 CSV metadata file for the final-checkpoint test confusion matrix
        pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(cm_csv)  # 3.12.26 save the final-checkpoint test confusion matrix as labeled CSV
        pd.DataFrame([{"group": str(model_name), "split": "test", "snapshot": "final", "n_runs": int(n_runs), "class_names": "|".join([str(x) for x in class_names])}]).to_csv(meta_csv, index=False)  # 3.12.26 save the final-checkpoint test confusion-matrix metadata as CSV

        cm_manifest_rows.append({  # 3.12.26 add this saved final-checkpoint test confusion matrix to the manifest table
            "group": str(model_name),  # 3.12.26 model group name
            "split": "test",  # 3.12.26 evaluation split name
            "snapshot": "final",  # 3.12.26 final checkpoint
            "n_runs": int(n_runs),  # 3.12.26 number of contributing runs
            "cm_csv": str(cm_csv),  # 3.12.26 path to the confusion-matrix CSV
            "meta_csv": str(meta_csv),  # 3.12.26 path to the metadata CSV
        })

if "conf_mats" in globals():  # 3.12.26 optionally save any older per-evaluation confusion matrices too
    legacy_dir = cm_dir / "legacy_conf_mats"  # 3.12.26 folder for older per-evaluation confusion-matrix exports
    legacy_dir.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the legacy confusion-matrix folder exists

    legacy_manifest_rows = []  # 3.12.26 one row per saved legacy confusion matrix

    for eval_key, cm in conf_mats.items():  # 3.12.26 walk through the older per-evaluation confusion-matrix dictionary
        if isinstance(eval_key, tuple) and (len(eval_key) == 4):  # 3.12.26 unpack the common tuple schema when available
            model_name, split_seed, trial_idx, init_seed = eval_key  # 3.12.26 parse the key into readable fields
        else:
            model_name, split_seed, trial_idx, init_seed = str(eval_key), 0, 0, 0  # 3.12.26 fallback when the key does not use the expected tuple format

        cm = np.asarray(cm, dtype=np.int64)  # 3.12.26 normalize the legacy confusion matrix to integer NumPy format
        stem = f"{_safe_name(model_name)}_split{int(split_seed)}_trial{int(trial_idx)}_init{int(init_seed)}"  # 3.12.26 filename stem for this legacy confusion matrix
        cm_csv = legacy_dir / f"{stem}_cm.csv"  # 3.12.26 CSV file for this legacy confusion matrix
        meta_csv = legacy_dir / f"{stem}_meta.csv"  # 3.12.26 CSV metadata file for this legacy confusion matrix
        pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(cm_csv)  # 3.12.26 save the legacy confusion matrix as labeled CSV
        pd.DataFrame([{"group": str(model_name), "split_seed": int(split_seed), "trial_index": int(trial_idx), "init_seed": int(init_seed), "class_names": "|".join([str(x) for x in class_names])}]).to_csv(meta_csv, index=False)  # 3.12.26 save the legacy confusion-matrix metadata as CSV
        legacy_manifest_rows.append({"group": str(model_name), "split_seed": int(split_seed), "trial_index": int(trial_idx), "init_seed": int(init_seed), "cm_csv": str(cm_csv), "meta_csv": str(meta_csv)})  # 3.12.26 add this legacy confusion matrix to its manifest

    pd.DataFrame(legacy_manifest_rows).to_csv(legacy_dir / "legacy_conf_mats_manifest.csv", index=False)  # 3.12.26 save the legacy confusion-matrix manifest as CSV
    print("Saved legacy confusion-matrix CSV files under:", legacy_dir)  # 3.12.26 confirm the legacy confusion-matrix save

if len(cm_manifest_rows) > 0:  # 3.12.26 save the main confusion-matrix manifest only when new-schema confusion matrices were written
    cm_manifest_csv = cm_dir / "confusion_matrix_manifest.csv"  # 3.12.26 CSV manifest for the new-schema aggregated confusion matrices
    pd.DataFrame(cm_manifest_rows).to_csv(cm_manifest_csv, index=False)  # 3.12.26 write the aggregated confusion-matrix manifest
    print("Saved confusion-matrix manifest CSV:", cm_manifest_csv)  # 3.12.26 confirm the confusion-matrix manifest save
else:
    print("Note: no new-schema confusion matrices were found to save.")  # 3.12.26 explain why the main confusion-matrix manifest may be missing

rep_dir = OUT_DIR / "reports"  # 3.12.26 folder for any legacy classification-report outputs
rep_dir.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the reports folder exists

if HAS_REPORTS:  # 3.12.26 save classification reports only when they exist
    reports_txt = rep_dir / "classification_reports.txt"  # 3.12.26 text export of the raw reports dictionary
    with open(reports_txt, "w") as f:  # 3.12.26 write one report after another in plain text
        for eval_key, report_text in reports.items():
            f.write("=" * 80 + "\n")
            f.write(f"EVAL KEY: {eval_key}\n")
            f.write(str(report_text) + "\n\n")
    print("Saved classification reports TXT:", reports_txt)  # 3.12.26 confirm the classification-report text save

    report_rows = []  # 3.12.26 long-format CSV version of the reports dictionary
    for eval_key, report_text in reports.items():  # 3.12.26 flatten the reports dictionary into rows that are easier to inspect than JSON
        report_rows.append({"eval_key": str(eval_key), "report_text": str(report_text)})  # 3.12.26 one row per saved report
    reports_csv = rep_dir / "classification_reports.csv"  # 3.12.26 CSV export of the raw reports dictionary
    pd.DataFrame(report_rows).to_csv(reports_csv, index=False)  # 3.12.26 save the raw reports as CSV
    print("Saved classification reports CSV:", reports_csv)  # 3.12.26 confirm the classification-report CSV save
else:
    print("Note: reports not found — skipping classification report export.")  # 3.12.26 explain why the reports folder may stay empty

fig_snap = OUT_DIR / "figures_snapshot"  # 3.12.26 folder where Cell 8 figures will be copied
fig_snap.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the figure-snapshot folder exists

copied_figure_rows = []  # 3.12.26 keep a CSV record of which figures were copied into the snapshot
if "FIG_OUTDIR" in globals() and (FIG_OUTDIR is not None):  # 3.12.26 copy figures only when Cell 8 already defined a figure directory
    src = Path(FIG_OUTDIR)  # 3.12.26 source figure directory from Cell 8
    if src.exists() and src.is_dir():  # 3.12.26 copy only if the source figure directory actually exists
        for p in sorted(src.glob("*")):  # 3.12.26 walk through the top-level figure files made by Cell 8
            if p.is_file() and p.suffix.lower() in [".png", ".svg", ".pdf"]:  # 3.12.26 copy common figure-file types
                dst = fig_snap / p.name  # 3.12.26 destination file path inside the snapshot folder
                dst.write_bytes(p.read_bytes())  # 3.12.26 simple copy without needing an extra shutil import
                copied_figure_rows.append({"source_path": str(p), "copied_path": str(dst), "suffix": str(p.suffix.lower())})  # 3.12.26 add this copied figure to the manifest
if len(copied_figure_rows) > 0:  # 3.12.26 save a manifest only when at least one figure was copied
    fig_manifest_csv = fig_snap / "figures_manifest.csv"  # 3.12.26 CSV manifest of copied Cell 8 figures
    pd.DataFrame(copied_figure_rows).to_csv(fig_manifest_csv, index=False)  # 3.12.26 write the copied-figure manifest as CSV
    print("Copied Cell 8 figures into:", fig_snap)  # 3.12.26 confirm the figure copy
else:
    print("Note: no Cell 8 figures were copied.")  # 3.12.26 explain why the figure snapshot folder may be empty

art_dir = OUT_DIR / "artifacts"  # 3.12.26 folder for artifact-folder manifests from Cell 7 trial outputs
art_dir.mkdir(parents=True, exist_ok=True)  # 3.12.26 ensure the artifacts folder exists

artifact_dirs = sorted({str(x) for x in df_summary["artifacts_dir"].dropna().astype(str).tolist()}) if ("artifacts_dir" in df_summary.columns) else []  # 3.12.26 collect the unique per-trial artifact directories referenced by df_summary
artifact_rows = []  # 3.12.26 one manifest row per per-trial artifact directory
artifact_file_rows = []  # 3.12.26 one long-format row per file found under the artifact directories

for artifact_dir in artifact_dirs:  # 3.12.26 summarize every per-trial artifact directory referenced by the summary table
    p = Path(artifact_dir)  # 3.12.26 normalize this artifact directory to a Path object
    exists = p.exists() and p.is_dir()  # 3.12.26 whether this artifact directory actually exists on disk
    n_files_recursive = int(sum(1 for q in p.rglob("*") if q.is_file())) if exists else 0  # 3.12.26 count how many files are inside this artifact directory
    artifact_rows.append({"artifacts_dir": str(p), "exists": bool(exists), "n_files_recursive": int(n_files_recursive)})  # 3.12.26 one summary row for this artifact directory

    if exists:  # 3.12.26 also record the individual files when the artifact directory exists
        for q in sorted(p.rglob("*")):  # 3.12.26 walk recursively through the artifact directory
            if q.is_file():
                artifact_file_rows.append({"artifacts_dir": str(p), "relative_file": str(q.relative_to(p)), "absolute_file": str(q)})  # 3.12.26 one long-format row per artifact file

artifact_dirs_csv = art_dir / "artifact_directories.csv"  # 3.12.26 summary CSV for the artifact directories
artifact_files_csv = art_dir / "artifact_files.csv"  # 3.12.26 long-format CSV listing the artifact files when they exist
pd.DataFrame(artifact_rows).to_csv(artifact_dirs_csv, index=False)  # 3.12.26 save the artifact-directory summary CSV
pd.DataFrame(artifact_file_rows).to_csv(artifact_files_csv, index=False)  # 3.12.26 save the artifact-file listing CSV
print("Saved artifact manifest CSV files under:", art_dir)  # 3.12.26 confirm the artifact-manifest saves

print("\nFinished saving evaluation outputs.")  # 3.12.26 final confirmation that this snapshot cell completed